# The Sorted: Mental Health App Data exploration and analysis

## 0.1 Imports


In [1]:
# Import necessary libraries
import json as json_lib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler


## 0.2 File Paths


In [2]:
#calling files
RESPONSE_PATH  = "U:/s2608480/Original csv/responses-25-12-19-at-06-37pm.csv"
TRACKS_PATH    = "U:/s2608480/Original csv/tracks-25-12-19-at-05-38pm.csv"
CODES_PATH     = "U:/s2608480/Original csv/referral_code_types.xlsx"

## 0.3 Load Raw Data


In [3]:
#loading files
response  = pd.read_csv(RESPONSE_PATH,  low_memory=False)
tracks  = pd.read_csv(TRACKS_PATH,    low_memory=False)
catagories_raw = pd.read_excel(CODES_PATH, header=None, names=['code_id','code_category'])


## 0.4 Colour Definitions


In [4]:
#define colour palette for users
colour_users = '#24C955', '#C3D4C3', '#646464'

colour_map_users={
    'Listener':            '#24C955',
    'Partial engager':    '#C3D4C3',
    'Ghost user':      '#646464',
}


colour_map_gender = {
    'Female':              '#E8748A',
    'Male':                '#378ADD',
    'Non-binary / Other':  '#9B59B6',
    'Prefer not to say':   '#888780',
    'Not provided':        '#D3D3D3',
}

colour_map_ethnicity = {
    'White':                          '#378ADD',
    'Asian':                          '#1D9E75',
    'African':                        '#EF9F27',
    'Caribbean or Black':             '#E8748A',
    'Mixed or multiple ethnic group': '#9B59B6',
    'Other ethnic group':             '#F39C12',
    'Prefer not to say':              '#888780',
    'Not provided':                   '#D3D3D3',
}

# define colour palette for referral codes
color_map_referral_codes = {
    'PURCHASE':  '#F75050',
    'PROMOTION': '#FCFF40',
    'NHS':       '#5F84FD',
    'FREE':      '#72C0DF',
    'EMPLOYER':  '#934FA1',
    'EDUCATION': '#20832E',
    'CHARITY':   '#FFAE35'
}

GREEN = '#24C955'
TEAL  = "#1D9E75"
BLUE  = "#378ADD"
AMBER = "#EF9F27"
GREY  = "#888780"
RED = "#D94F3D"


---
# 1. Data Preparation


## 1.1 Column Standardisation, Type Casting & Datetime Parsing


In [5]:
# Column cleaning
def clean_cols(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
    return df

# Data types
response = clean_cols(response); tracks = clean_cols(tracks)
for col in ['id','code_id']:
    for df in [response,tracks]:
        if col in df.columns: df[col] = df[col].astype('string')

# Datetimes
response['user_created_on_time'] = pd.to_datetime(
    response['user_created_on_time'], format='%y-%m-%d-at-%I-%M%p', errors='coerce')
for col in ['last_open_time','open_time']:
    if col in response.columns: response[col] = pd.to_datetime(response[col], errors='coerce')
response['created_date']    = response['user_created_on_time'].dt.normalize()
response['created_hour']    = response['user_created_on_time'].dt.hour
response['created_weekday'] = response['user_created_on_time'].dt.day_name()

# Tracks datetimes
dt = pd.to_datetime(tracks['datetimeoflistening'], format='%y-%m-%d-at-%I-%M%p', errors='coerce')
tracks['datetimeoflistening'] = dt
tracks['listening_date']    = dt.dt.normalize()
tracks['listening_hour']    = dt.dt.hour
tracks['listening_weekday'] = dt.dt.day_name()

# progresspercent
def clean_pp(x):
    if pd.isna(x): return pd.NA
    if isinstance(x,(int,float)): return int(x)
    if isinstance(x,str):
        x=x.strip()
        if x.isdigit(): return int(x)
        if x.startswith('{'):
            try:
                d=json_lib.loads(x)
                if 'value' in d: return int(d['value'])
            except: return pd.NA
    return pd.NA

# Clean progresspercent column
tracks['progresspercent_clean'] = tracks['progresspercent'].apply(clean_pp).astype('Int64')

tracks['trackduration']         = pd.to_numeric(tracks['trackduration'], errors='coerce')
tracks['dayssincefirstopen']    = pd.to_numeric(tracks['dayssincefirstopen'], errors='coerce')

## 1.2 Referral Code Merge & Exclusion of Non-Clinical Categories


In [6]:
# Exclusion set
EXCLUDE = {'INTERNAL','RESEARCH','MARKETTING'}

# Read in the data
response = response.merge(catagories_raw, on='code_id', how='left')
response = response[~response['code_category'].str.upper().isin(EXCLUDE)].copy()
response['user_number'] = (pd.factorize(response['id'])[0]+1).astype(str)

# Merge the response data with the tracks data to get the user_number and code_category for each track
tracks = tracks.merge(response[['id','code_category','user_number']], on='id', how='left')
max_u = int(tracks['user_number'].dropna().astype(int).max())
missing = tracks.loc[tracks['user_number'].isna(),'id'].unique()
new_map = {k:str(i) for i,k in enumerate(missing, max_u+1)}
tracks.loc[tracks['user_number'].isna(),'user_number'] =     tracks.loc[tracks['user_number'].isna(),'id'].map(new_map)
tracks = tracks[~tracks['code_category'].str.upper().isin(EXCLUDE)].copy()


## 1.3 Demographic Cleaning


In [7]:
# Age
response['age'] = response['age'].astype('string').str.strip()
age_map = {'12-15':'12–15','15-24':'16–24','16-24':'16–24','25-64':'25–64',
           '65 & over':'65+','65 and over':'65+','65 i powyżej':'65+'}
response['age_clean'] = response['age'].map(age_map)
response.loc[response['age'].isin(['100','10000']),'age_clean'] = pd.NA
response['age_clean'] = pd.Categorical(response['age_clean'],
    categories=['12–15','16–24','25–64','65+'], ordered=True)

# Gender
response['gender'] = response['gender'].astype('string').str.strip().str.lower()
gmap = {
    'female':'Female','f':'Female','woman':'Female','kobieta':'Female',
    'male':'Male','m':'Male','man':'Male','mężczyzna':'Male','my sex is male':'Male',
    'non-binary':'Non-binary / Other','non binary':'Non-binary / Other',
    'nonbinary':'Non-binary / Other','genderfluid':'Non-binary / Other',
    'genderqueer':'Non-binary / Other','gender fluid':'Non-binary / Other',
    'other/non-binary/multiple genders/genderfluid':'Non-binary / Other',
    'agender/none':'Non-binary / Other','other':'Non-binary / Other',
    'she/they':'Non-binary / Other','they/them':'Non-binary / Other',
    'osoba apłciowa':'Non-binary / Other','demigirl':'Non-binary / Other',
    'demi-girl':'Non-binary / Other','gay':'Non-binary / Other',
    'prefer not to say':'Prefer not to say','prefer not say':'Prefer not to say',
    'prefer not to':'Prefer not to say','rather not say':'Prefer not to say',
    "don't want to say":'Prefer not to say','no comment':'Prefer not to say',
    'none of your business':'Prefer not to say','no':'Prefer not to say',
    'na':'Prefer not to say','.':'Prefer not to say',
    'test':'TEST','testing':'TEST','testtest':'TEST','apple1test':'TEST',
}
response['gender_clean'] = response['gender'].map(gmap)
response.loc[response['gender'].notna()&response['gender_clean'].isna(),'gender_clean']='Non-binary / Other'
response['gender_clean'] = pd.Categorical(response['gender_clean'],
    categories=['Female','Male','Non-binary / Other','Prefer not to say'])

# Ethnicity (condensed 4-pass)
response['ethnic_group_raw'] = response['ethnic_group'].astype('string').str.strip().str.lower()
def _eth(x):
    if pd.isna(x): return pd.NA
    x=x.lower()
    if 'prefer not' in x or 'rather not' in x: return 'Prefer not to say'
    if any(t in x for t in ['white','scottish','british','irish','polish','roma',
                             'gypsy','traveller','european','german','italian','spanish',
                             'french','greek','dutch','swedish','american','canadian',
                             'australian','russian','ukrainian','nordic']): return 'White'
    if any(t in x for t in ['asian','pakistani','indian','bangladeshi','chinese',
                             'filipino','turkish','japanese','korean','thai',
                             'vietnamese','sri lanka']): return 'Asian'
    if 'african' in x: return 'African'
    if 'caribbean' in x or 'black' in x or 'coloured' in x: return 'Caribbean or Black'
    if 'mixed' in x or 'multiple' in x: return 'Mixed or multiple ethnic group'
    if any(t in x for t in ['arab','jewish','sikh','other']): return 'Other ethnic group'
    return 'Other ethnic group'
response['ethnic_group_clean'] = response['ethnic_group_raw'].apply(_eth)
junk = ['none','no','na','n/a','unsure','xxxx','x','-','no comment','non','k','']
response.loc[response['ethnic_group_raw'].isin(junk),'ethnic_group_clean'] = pd.NA
response.loc[response['ethnic_group_raw'].str.contains('test',case=False,na=False),
         'ethnic_group_clean'] = 'TEST ACCOUNT'

# Remove test accounts entirely
test_mask = ((response['gender_clean'].astype(str).str.lower()=='test') |
             (response['ethnic_group_clean'].astype(str)=='TEST ACCOUNT'))
n_test = test_mask.sum()
response = response[~test_mask].reset_index(drop=True)
tracks = tracks[tracks['id'].isin(response['id'])].copy()

## 1.4 Clinical Change Scores


In [8]:
# Define the timepoints for each scoring instrument
SCORE_TIMEPOINTS = {
    'PHQ-8':     ['q_phq8_0','q_phq8_2','q_phq8_4','q_phq8_6','q_phq8_8'],
    'GAD-7':     ['q_gad7_0','q_gad7_2','q_gad7_4','q_gad7_6','q_gad7_8'],
    'Psychlops': ['q_psyclops_0','q_psyclops_2','q_psyclops_4','q_psyclops_6','q_psyclops_8'],
}
ONS4_COLS = sorted([c for c in response.columns if 'ons4' in c.lower()])
if ONS4_COLS:
    SCORE_TIMEPOINTS['ONS-4'] = ONS4_COLS

LOWER_IS_BETTER = {'PHQ-8', 'GAD-7', 'Psychlops'}
HIGHER_IS_BETTER = {'ONS-4'}

# Compute clinical change scores:

def compute_change(df, name, cols):
    present = [c for c in cols if c in df.columns]
    if len(present) < 2:
        return pd.Series(np.nan, index=df.index, dtype=float)

    for c in present:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    baseline_col = present[0]
    followup_cols = present[1:]
    baseline = df[baseline_col]

    # Latest available follow-up only; if no follow-up exists, change remains missing.
    latest_followup = df[followup_cols].apply(
        lambda r: r.dropna().iloc[-1] if r.dropna().shape[0] > 0 else np.nan,
        axis=1
    )

    if name in LOWER_IS_BETTER:
        return latest_followup - baseline
    if name in HIGHER_IS_BETTER:
        return baseline - latest_followup

    return latest_followup - baseline

for name, cols in SCORE_TIMEPOINTS.items():
    response[f'change_{name}'] = compute_change(response, name, cols)

# Add the change columns to the list used later for listener-level enrichment.
CHANGE_COLS = [f'change_{n}' for n in SCORE_TIMEPOINTS]

# Backwards-compatible alias for later cells that still refer to IMP_COLS.
IMP_COLS = CHANGE_COLS  


---
# 2. User Segmentation


In [9]:
# Categorize users based on their listening behavior
listened_ids = set(response['id']) & set(tracks['id'])
never_ids    = set(response['id']) - set(tracks['id'])

# Create separate DataFrames for listeners and non-listeners
listeners_response  = response[response['id'].isin(listened_ids)].reset_index(drop=True)
listeners_tracks    = tracks[tracks['id'].isin(listened_ids)].reset_index(drop=True)
non_listeners       = response[response['id'].isin(never_ids)].reset_index(drop=True)

# Categorize non-listeners into those who provided some responses and those who provided none
response_cols = [c for c in ['q_phq8_0','q_gad7_0','q_psyclops_0',
    'would_use_again2','tracks_listened0','mood'] if c in non_listeners.columns]
ghost_mask = non_listeners[response_cols].isnull().all(axis=1) if response_cols     else pd.Series([True]*len(non_listeners))
non_listeners_empty = non_listeners[ghost_mask].reset_index(drop=True)
non_listeners_some  = non_listeners[~ghost_mask].reset_index(drop=True)

total = len(response)
print(f"{'Listeners'} {len(listeners_response)}")
print(f"{'Partial engagers'} {len(non_listeners_some)}")
print(f"{'Ghost users'} {len(non_listeners_empty)}")
print(f"{'Listening events'} {len(listeners_tracks)}")


Listeners 31282
Partial engagers 18204
Ghost users 43836
Listening events 463131


---
# 3. Feature Engineering


## 3.1 Programme Mapping, Track Features & Per-User Engagement Summary


In [10]:

TRACKID_TO_PROGRAMME = {
    'intro':'Welcome','welcome':'Welcome','welcome_2025':'Welcome',
    'guided_body_relaxation':'Feeling Good for Life','mindful_body_scan':'Feeling Good for Life',
    'inner_safe_place':'Feeling Good for Life','access_deep_calm':'Feeling Good for Life',
    'self_confidence':'Feeling Good for Life','meet_challenges':'Feeling Good for Life',
    'link_mind_body':'Feeling Good for Life','trigger_the_future':'Feeling Good for Life',
    'distance_meaning':'Feeling Good for Life','increase_self_esteem':'Feeling Good for Life',
    'think_creatively':'Feeling Good for Life','visualise_performance':'Feeling Good for Life',
    'inner_advisor':'Feeling Good for Life','view_from_mountain':'Feeling Good for Life',
    'guided_body_relaxation_2024':'Feeling Good for Life',
    'mindfulness_in_3':'Booster Tracks','relaxation_boost':'Booster Tracks',
    'confidence_boost':'Booster Tracks',
    'sleep_soundly':'Sleep Better','restful_night':'Sleep Better',
    'positive_ageing_1':'Positive Ageing','positive_ageing_2':'Positive Ageing',
    'quit_smoking':'Stop Smoking','stay_stopped':'Stop Smoking',
    'goal_healthy_weight':'Healthy Body Weight','love_food_body':'Healthy Body Weight',
    'lose_weight':'Healthy Body Weight','love_your_food_and_body':'Healthy Body Weight',
    'long_covid_intro':'Help for Long Covid','body_and_mind':'Help for Long Covid',
    'immune_body':'Help for Long Covid',
    'ou_distance_meaning':'OU Work Life Balance','ou_access_deep_calm':'OU Exam Stress',
    'ou_link_mind_body':'OU Exercise Motivation','ou_self_confidence':'OU Feel More Decisive',
    **{f'guided_body_relaxation_punjabi':'Feeling Good Punjabi',
       'mindful_body_scan_punjabi':'Feeling Good Punjabi',
       'inner_safe_place_punjabi':'Feeling Good Punjabi',
       'access_deep_calm_punjabi':'Feeling Good Punjabi',
       'self_confidence_punjabi':'Feeling Good Punjabi',
       'meet_challenges_punjabi':'Feeling Good Punjabi',
       'link_mind_body_punjabi':'Feeling Good Punjabi',
       'trigger_the_future_punjabi':'Feeling Good Punjabi',
       'distance_meaning_punjabi':'Feeling Good Punjabi',
       'increase_self_esteem_punjabi':'Feeling Good Punjabi',
       'think_creatively_punjabi':'Feeling Good Punjabi',
       'visualise_performance_punjabi':'Feeling Good Punjabi'},
    **{'guided_body_relaxation_japan':'Feeling Good Japan',
       'relaxation_boost_japan':'Feeling Good Japan',
       'confidence_boost_japan':'Feeling Good Japan',
       'mindfulness_in_3_japan':'Feeling Good Japan'},
    **{f'teens_{i}_track':'Teens' for i in range(1,12)},
    **{k:'Monthly Snippets' for k in [
        'dec22_importance_of_sleep','jan23_sports_and_mental_health','feb23_burn_out',
        'mar23_breathing','apr23_student_stress','may23_exam_stress','june2023_loneliness',
        'july2023_active_over_summer','august2023_back_to_school',
        'september2023_pursuit_of_happiness','october2023_imposter_syndrome',
        'november2023_dynamic_balance_of_our_emotions',
        'december2023_bryony_mental_health_advocacy','nov22_burn_out']},
}
TRACK_ORDER = {
    'intro':1,'welcome':2,
    'guided_body_relaxation':1,'mindful_body_scan':2,'inner_safe_place':3,
    'access_deep_calm':4,'self_confidence':5,'meet_challenges':6,
    'link_mind_body':7,'trigger_the_future':8,'distance_meaning':9,
    'increase_self_esteem':10,'think_creatively':11,'visualise_performance':12,
    'inner_advisor':13,'view_from_mountain':14,
    'mindfulness_in_3':1,'relaxation_boost':2,'confidence_boost':3,
    'sleep_soundly':4,'restful_night':5,'positive_ageing_1':4,'positive_ageing_2':5,
    'quit_smoking':4,'stay_stopped':6,'goal_healthy_weight':4,'love_food_body':5,
    'long_covid_intro':1,'body_and_mind':2,'immune_body':3,
    **{f'teens_{i}_track':i for i in range(1,12)},
}

# Function to categorize listening hours
def bin_hour(h):
    if pd.isna(h): return pd.NA
    if 5<=h<12: return 'Morning'
    if 12<=h<17: return 'Afternoon'
    if 17<=h<21: return 'Evening'
    return 'Night'

listeners_tracks = listeners_tracks[listeners_tracks['trackid'].notna()].reset_index(drop=True)
listeners_tracks['time_of_day']    = listeners_tracks['listening_hour'].apply(bin_hour)
listeners_tracks['programme_name'] = listeners_tracks['moduleid'].fillna(
    listeners_tracks['trackid'].map(TRACKID_TO_PROGRAMME))
listeners_tracks['track_position'] = listeners_tracks['trackid'].map(TRACK_ORDER)

# Function to check sequential adherence
def sequential_adherence(group):
    for _,pg in group.groupby('programme_name'):
        pg = pg.sort_values('datetimeoflistening')
        positions = pg['track_position'].dropna().tolist()
        found = []
        for p in positions:
            if p==len(found)+1 and p<=3: found.append(p)
        if found==[1,2,3]: return True
    return False

# Function to summarise user listening behaviour
def summarise_user(group):
    days_sorted = sorted(group['datetimeoflistening'].dropna())
    if len(days_sorted)>=2:
        gaps = [(days_sorted[i+1]-days_sorted[i]).days for i in range(len(days_sorted)-1)]
        regularity = np.std(gaps)
    else: regularity = np.nan
    tod  = group['time_of_day'].value_counts()
    prog = group['programme_name'].value_counts()
    return pd.Series({
        'total_tracks':         len(group),
        'mean_completion':      group['progresspercent_clean'].mean(),
        'full_completion_rate': (group['progresspercent_clean']==100).sum()/len(group),
        'programmes_explored':  group['programme_name'].nunique(),
        'days_active':          group['listening_date'].nunique(),
        'listening_span_days':  group['dayssincefirstopen'].max()-group['dayssincefirstopen'].min(),
        'regularity':           regularity,
        'preferred_time_of_day':tod.idxmax() if len(tod)>0 else pd.NA,
        'booster_track_user':   (group['programme_name']=='Booster Tracks').any(),
        'sequential_adherence': sequential_adherence(group),
        'top_programme':        prog.idxmax() if len(prog)>0 else pd.NA,
    })

user_summary = (
    listeners_tracks
    .groupby("id")
    .apply(summarise_user, include_groups=False)
    .reset_index()
)
for col in ["mean_completion", "full_completion_rate", "regularity"]:
    user_summary[col] = pd.to_numeric(user_summary[col], errors="coerce").round(3)

# Enrich listeners_response with user_summary
listeners_enriched = listeners_response.merge(user_summary, on="id", how="left")
for col in IMP_COLS:
    if col in listeners_response.columns:
        listeners_enriched[col] = listeners_response.set_index('id').loc[
            listeners_enriched['id'], col].values


## 3.2 Clinical Change Scores & Responder Flags (Listener-Level)


In [11]:
# clinical outcome specifications for each measure

OUTCOME_SPECS = {
    'GAD-7': {
        'baseline': 'q_gad7_0',
        'followups': ['q_gad7_8', 'q_gad7_6', 'q_gad7_4', 'q_gad7_2'],
        'direction': 'lower_better',
        'mcid': 5,
        'prefix': 'gad7'
    },
    'PHQ-8': {
        'baseline': 'q_phq8_0',
        'followups': ['q_phq8_8', 'q_phq8_6', 'q_phq8_4', 'q_phq8_2'],
        'direction': 'lower_better',
        'mcid': 5,
        'prefix': 'phq8'
    },
    'Psychlops': {
        'baseline': 'q_psyclops_0',
        'followups': ['q_psyclops_8', 'q_psyclops_6', 'q_psyclops_4', 'q_psyclops_2'],
        'direction': 'lower_better',
        'mcid': None,
        'prefix': 'psychlops'
    }
}

# Add ONS-4 if at least two columns are present
ons4_cols = [c for c in listeners_enriched.columns if 'ons4' in c.lower()]
if len(ons4_cols) >= 2:
    def _suffix_num(col):
        try:
            return int(str(col).split('_')[-1])
        except Exception:
            return 999

    ons4_cols_sorted = sorted(ons4_cols, key=_suffix_num)
    OUTCOME_SPECS['ONS-4'] = {
        'baseline': ons4_cols_sorted[0],
        'followups': list(reversed(ons4_cols_sorted[1:])),
        'direction': 'higher_better',
        'mcid': None,
        'prefix': 'ons4'
    }


def latest_followup(row, followup_cols):
    """Return the latest available follow-up value and its column name."""
    for col in followup_cols:
        if col in row.index and pd.notna(row[col]):
            return row[col], col
    return np.nan, None


def _change_score(baseline, followup, direction):
    """Return a change score where negative values indicate improvement."""
    if direction == 'lower_better':
        return followup - baseline
    if direction == 'higher_better':
        return baseline - followup
    return followup - baseline


def _tp_label(col):
    """Convert q_gad7_2 style names to T2 for timepoint-specific columns."""
    try:
        return f"T{int(str(col).split('_')[-1])}"
    except Exception:
        return str(col).replace('q_', '').replace('_', '').upper()


for measure, spec in OUTCOME_SPECS.items():
    baseline_col = spec['baseline']
    followup_cols = [c for c in spec['followups'] if c in listeners_enriched.columns]
    prefix = spec['prefix']

    if baseline_col not in listeners_enriched.columns or len(followup_cols) == 0:
        print(f"Skipped {measure}: required columns not found")
        continue

    listeners_enriched[baseline_col] = pd.to_numeric(listeners_enriched[baseline_col], errors='coerce')
    for col in followup_cols:
        listeners_enriched[col] = pd.to_numeric(listeners_enriched[col], errors='coerce')

        tp = _tp_label(col)
        listeners_enriched[f'{prefix}_change_{tp}'] = _change_score(
            listeners_enriched[baseline_col],
            listeners_enriched[col],
            spec['direction']
        )

    followup_result = listeners_enriched.apply(
        lambda r: latest_followup(r, followup_cols),
        axis=1
    )

    listeners_enriched[f'{measure}_followup'] = [x[0] for x in followup_result]
    listeners_enriched[f'{measure}_followup_tp'] = [x[1] for x in followup_result]

    change = _change_score(
        listeners_enriched[baseline_col],
        listeners_enriched[f'{measure}_followup'],
        spec['direction']
    )

    listeners_enriched[f'{measure}_change'] = change
    listeners_enriched[f'{prefix}_change'] = change
    listeners_enriched[f'{prefix}_followup'] = listeners_enriched[f'{measure}_followup']
    listeners_enriched[f'{prefix}_followup_tp'] = listeners_enriched[f'{measure}_followup_tp']
    listeners_enriched[f'{measure}_improvement_magnitude'] = -change
    listeners_enriched[f'{prefix}_improvement_magnitude'] = -change

    if spec['mcid'] is not None:
        listeners_enriched[f'{measure}_responder'] = change <= -spec['mcid']
        listeners_enriched[f'{prefix}_responder'] = change <= -spec['mcid']


coverage_rows = []
for measure, spec in OUTCOME_SPECS.items():
    prefix = spec['prefix']
    change_col = f'{prefix}_change'
    if change_col in listeners_enriched.columns:
        vals = listeners_enriched[change_col].dropna()
        coverage_rows.append({
            'measure': measure,
            'n_with_change': int(vals.shape[0]),
            'pct_of_listeners': round(vals.shape[0] / len(listeners_enriched) * 100, 1),
            'mean_change': round(vals.mean(), 2),
            'median_change': round(vals.median(), 2),
            'direction': 'negative = improvement'
        })

coverage = pd.DataFrame(coverage_rows)


---
# 4. Sustained Engager Definition


In [12]:
# Parse listen_date if not already present
if 'listen_date' not in listeners_tracks.columns:
    listeners_tracks['listen_date'] = pd.to_datetime(
        listeners_tracks['datetimeoflistening'],
        format='%y-%m-%d-at-%I-%M%p',
        errors='coerce'
    )

# Compute distinct active weeks per user
weeks_active = (
    listeners_tracks.assign(
        week_key=listeners_tracks['listen_date'].dt.isocalendar().week.astype(str)
                 + '-' +
                 listeners_tracks['listen_date'].dt.year.astype(str)
    )
    .groupby('id')['week_key']
    .nunique()
    .reset_index(name='weeks_active')
)

listeners_enriched = listeners_enriched.merge(weeks_active, on='id', how='left')
listeners_enriched['weeks_active'] = listeners_enriched['weeks_active'].fillna(0).astype(int)

# Define sustained engagers
sustained_mask = (
    (listeners_enriched['listening_span_days'] > 30) &
    (listeners_enriched['weeks_active'] > 1)
)

sustained_users = listeners_enriched[sustained_mask].copy()
other_listeners = listeners_enriched[~sustained_mask].copy()

---
# 5. Exploratory Analysis


## 5.1 Population Overview


In [13]:
# Create the grid
n_list=len(listeners_response); n_some=len(non_listeners_some)
n_ghost=len(non_listeners_empty); total_w=n_list+n_some+n_ghost
pct_l=round(n_list/total_w*100); pct_s=round(n_some/total_w*100)
pct_e=100-pct_l-pct_s
cells_grid=([0]*pct_l+[1]*pct_s+[2]*pct_e)[:100]+[2]*(100-min(100,pct_l+pct_s+pct_e))
grid=np.array(cells_grid[:100]).reshape(10,10)

# set colours
GROUPS=[('Listeners',n_list,'#24C955'),('Partial engagers',n_some,'#C3D4C3'),('Ghost users',n_ghost,'#646464')]
fig=go.Figure()

# Add the squares to the grid
for row in range(10):
    for col in range(10):
        v=grid[row,col]
        fig.add_shape(type='rect',x0=col+0.05,x1=col+0.87,y0=9-row+0.05,y1=9-row+0.87,
            fillcolor=colour_users[v],line_color='white',line_width=2)

# Add the legend
for lbl,n,col in GROUPS:
    fig.add_trace(go.Scatter(x=[None],y=[None],mode='markers',
        marker=dict(size=14,color=col,symbol='square'),
        name=f'{lbl}  —  {n:,}  ({n/total_w*100:.1f}%)'))

# Update the layout
fig.update_layout(title=f'User Engagement Breakdown  ·  n={total_w:,}  ·  each square ≈ 1%',
    xaxis=dict(range=[0,10],showticklabels=False,showgrid=False,zeroline=False),
    yaxis=dict(range=[0,10],showticklabels=False,showgrid=False,zeroline=False),
    height=420,
    width=700,
    legend=dict(orientation='v',x=1.02,y=0.5),
    margin=dict(t=60,r=220),plot_bgcolor='white')
fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

## 5.2 Demographic Profiles — Listeners


In [14]:
# Calculate gender distribution
gender_counts = (
    listeners_enriched['gender_clean']
    .astype(object)
    .fillna('Not provided')
    .astype(str)
    .replace({'nan': 'Not provided', '<NA>': 'Not provided', 'null': 'Not provided'})
    .pipe(lambda s: s[s != 'TEST'])
    .value_counts(dropna=False)
    .reset_index()
)
gender_counts.columns = ['gender', 'count']

gender_counts['colour'] = gender_counts['gender'].map(colour_map_gender).fillna('#CCCCCC')

# Create the pie chart
fig = go.Figure(
    data=[
        go.Pie(
            labels=gender_counts['gender'],
            values=gender_counts['count'],
            marker=dict(colors=gender_counts['colour'].tolist()),
            textinfo='label+percent',
            hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>'
        )
    ]
)


fig.update_layout(
    title=dict(
        text='Gender Distribution of Listeners',
        x=0.5,
        xanchor='center',
        font=dict(size=22)
    ),
    legend=dict(
        orientation='v',
        x=1.02,
        y=0.5,
        font=dict(size=14),
        title=dict(text='Gender')
    ),
    margin=dict(t=80, b=40, l=40, r=180),
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=900,
    height=650
)

fig.show()


In [15]:
# Calculate ethnic group distribution
eth_counts_all = (
    response['ethnic_group_clean']
    .astype(object)
    .fillna('Not provided')
    .astype(str)
    .replace({'nan': 'Not provided', '<NA>': 'Not provided', 'null': 'Not provided'})
    .value_counts(dropna=False)
    .reset_index()
)
eth_counts_all.columns = ['ethnicity', 'count']

eth_counts_all['colour'] = eth_counts_all['ethnicity'].map(colour_map_ethnicity).fillna('#CCCCCC')

# Create a pie
fig2 = go.Figure(
    data=[
        go.Pie(
            labels=eth_counts_all['ethnicity'],
            values=eth_counts_all['count'],
            hole=0.4,
            marker=dict(colors=eth_counts_all['colour'].tolist()),
            textinfo='label+percent',
            hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>'
        )
    ]
)

# Update layout for the pie chart
fig2.update_layout(
    title=dict(text='Ethnic Group Distribution (All Users)', font=dict(size=18)),
    showlegend=True,
    legend=dict(orientation='v', x=1.05, y=0.5),
    margin=dict(t=60, b=20, l=20, r=150),
    paper_bgcolor='white'
)

fig2.show()

## 5.3 Installation Trends Over Time

In [16]:
# Rebuild all_resp with group labels
all_resp = pd.concat([
    listeners_response.assign(group='Listener'),
    non_listeners_some.assign(group='Partial engager'),
    non_listeners_empty.assign(group='Ghost user')
], ignore_index=True)

all_resp['created_date'] = pd.to_datetime(all_resp['created_date'], errors='coerce')

all_resp['month'] = all_resp['created_date'].dt.to_period('M').astype(str)
monthly = (all_resp.groupby(['month', 'group'])['id'].count()
           .unstack(fill_value=0).reset_index())
for col in ['Listener', 'Partial engager', 'Ghost user']:
    if col not in monthly.columns: monthly[col] = 0
monthly['total'] = monthly[['Listener', 'Partial engager', 'Ghost user']].sum(axis=1)
monthly['ghost_pct'] = (monthly['Ghost user'] / monthly['total'] * 100).round(1)
monthly['listener_pct'] = (monthly['Listener'] / monthly['total'] * 100).round(1)
monthly['month_dt'] = pd.to_datetime(monthly['month'], format='%Y-%m')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=['Monthly Installs (stacked)', 'Monthly Installs by Group'])  # ← fixed

for grp, col in [('Listener', '#24C955'), ('Partial engager', '#C3D4C3'), ('Ghost user', '#646464')]:
    fig.add_trace(go.Bar(name=grp, x=monthly['month'], y=monthly[grp],
        marker_color=col), row=1, col=1)  # ← fixed

fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['Listener'],
    mode='lines+markers', name='Listeners',
    line=dict(color='#24C955', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['Ghost user'],
    mode='lines+markers', name='Ghost users',
    line=dict(color='#646464', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['Partial engager'],
    mode='lines+markers', name='Partial engagers',
    line=dict(color='#C3D4C3', width=2)), row=2, col=1)

fig.update_yaxes(title_text='Number of users', row=2, col=1)

fig.update_layout(
    height=620,
    barmode='stack',
    title_text='Monthly Installs & Engagement Rates',
    margin=dict(t=70),
    xaxis2_tickangle=-45
)

fig.show()


### 5.3b Installation Heatmap by Routs of access


In [17]:
response['month'] = response['user_created_on_time'].dt.month_name()
response['month_num'] = response['user_created_on_time'].dt.month

heatmap_data = (
    response
    .groupby(['month', 'month_num', 'created_weekday'])
    .size()
    .reset_index(name='count')
)

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Create a pivot table for the heatmap
heatmap_pivot = heatmap_data.pivot_table(
    index='month',
    columns='created_weekday',
    values='count',
    aggfunc='sum'
)

# correct ordering
heatmap_pivot = heatmap_pivot.reindex(index=month_order, columns=day_order)

heatmap_pivot = heatmap_pivot.fillna(0)

# create heatmap
fig = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    colorscale='YlOrRd',
    colorbar=dict(title='Number of Installations'),
    text=heatmap_pivot.values,
    texttemplate="%{text:.0f}",
    hovertemplate='Month: %{y}<br>Day: %{x}<br>Installs: %{z}<extra></extra>'
))

fig.update_layout(
    title='App Installations by Month and Day of Week',
    xaxis_title='Day of Week',
    yaxis_title='',
    width=900,
    height=600
)

fig.show()

In [18]:
# Create a pivot table for the stacked bar chart
hourly_cat = (
    response.groupby(['created_hour', 'code_category'])
    .size()
    .reset_index(name='installs')
)

pivot = hourly_cat.pivot_table(
    index='created_hour',
    columns='code_category',
    values='installs',
    fill_value=0
)

categories = list(pivot.columns)

fig_bar = go.Figure()

# Add traces for each category
for cat in categories:
    fig_bar.add_trace(go.Bar(
        x=pivot.index,
        y=pivot[cat],
        name=cat, 
        marker=dict(color=color_map_referral_codes.get(cat, '#999999'))
    ))

# Working hours shading
fig_bar.add_vrect(
    x0=8.5, x1=17.5,
    fillcolor='grey',
    opacity=0.06,
    line_width=0,
    annotation_text='Working hours',
    annotation_position='top left'
)

# Update layout for the stacked bar chart
fig_bar.update_layout(
    barmode='stack',
    title='App Installations by Hour of Day and Access Type',
    xaxis_title='Hour of Day (0 = midnight)',
    yaxis_title='Number of Installations',
    xaxis=dict(tickmode='linear', dtick=1),
    template='plotly_white',
    width=1000,
    height=450,
    legend_title_text='Access Type'
)

fig_bar.show()

## 5.4 Demographics Across Engagement Groups


In [19]:
# Create subplots for each demographic category
frames={'Listeners':listeners_response,'Partial engagers':non_listeners_some,'Ghost users':non_listeners_empty}

# Create a bar chart for each demographic category
fig=make_subplots(rows=1,cols=3,subplot_titles=['Age','Gender','Code category'],horizontal_spacing=0.1)
for i,(label,df) in enumerate(frames.items()):
    if 'age_clean' in df.columns:
        age=df['age_clean'].astype(str).value_counts().reindex(['12–15','16–24','25–64','65+'],fill_value=0)
        fig.add_trace(go.Bar(name=label,x=age.index.tolist(),y=age.values,
            marker_color=colour_users[i],showlegend=True),row=1,col=1)
    if 'gender_clean' in df.columns:
        gen=df['gender_clean'].astype(str).value_counts().head(4)
        fig.add_trace(go.Bar(name=label,x=gen.index.tolist(),y=gen.values,
            marker_color=colour_users[i],showlegend=False),row=1,col=2)
    if 'code_category' in df.columns:
        cat=df['code_category'].value_counts().head(6)
        fig.add_trace(go.Bar(name=label,x=cat.index.tolist(),y=cat.values,
            marker_color=colour_users[i],showlegend=False),row=1,col=3)
fig.update_layout(height=420,barmode='group',title_text='Demographics by User Group',margin=dict(t=70))
fig.show()


In [20]:
# Create a bar chart for referral codes by engagement group
frames = {
    'Listener':        listeners_response,
    'Partial engager': non_listeners_some,
    'Ghost user':      non_listeners_empty,
}

all_cats = pd.concat(
    [df['code_category'] for df in frames.values() if 'code_category' in df.columns]
)
ordered_cats = all_cats.value_counts().head(7).index.tolist()

fig = go.Figure()

# ── Bars ──────────────────────────────────────────────────────────────────────
for label, df in frames.items():
    counts = df['code_category'].value_counts()
    y_vals = [counts.get(cat, 0) for cat in ordered_cats]

    fig.add_trace(go.Bar(
        name=label,
        x=ordered_cats,
        y=y_vals,
        marker_color=colour_map_users[label],
    ))


fig.update_layout(
    title_text='How Users Were Referred, by Engagement Group',
    barmode='group',
    height=440,
    width=750,
    margin=dict(t=70, r=40, b=80),
    plot_bgcolor='#F5F5F5',
    paper_bgcolor='white',
    xaxis=dict(
        tickmode='array',
        tickvals=ordered_cats,
        ticktext=[
            f'<span style="color:{"#B8960A" if cat == "PROMOTION" else color_map_referral_codes.get(cat, "#333333")}"><b>{cat}</b></span>'
            for cat in ordered_cats
        ],
        tickangle=-45,
        tickfont=dict(size=12, family='Arial'),
    ),
    legend=dict(orientation='v', x=1.02, y=0.5),
    bargap=0.25,
    bargroupgap=0.05,
)

fig.show()

---
# 6. Access Route Analysis


## 6.1 Installations by Access Route


In [21]:
# Summarize installation data by referral category
install_summary = (response.groupby('code_category')['id']
    .count().reset_index(name='total'))
install_summary['pct'] = (install_summary['total'] / install_summary['total'].sum() * 100).round(1)
install_summary = install_summary.sort_values('total', ascending=False)
install_summary['label'] = install_summary.apply(
    lambda r: f"{r['total']:,}<br>{r['pct']}%", axis=1
)

fig = go.Figure()

# Add bar trace for installations by access route
fig.add_trace(go.Bar(
    x=install_summary['code_category'],
    y=install_summary['total'],
    name='Installations',
    marker_color=install_summary['code_category'].map(color_map_referral_codes),
    text=install_summary['label'],
    textposition='outside'
))

fig.update_layout(
    title='Installations by Access Route',
    height=450,
    width=700,
    plot_bgcolor='#E8EDF4',
    margin=dict(t=70),
    xaxis_tickangle=-30,
    yaxis=dict(
        title='Number of installations',
        range=[0, install_summary['total'].max() * 1.25]
    ),
    showlegend=False
)

fig.show()


## 6.2 Ghost Rate by Access Route


In [22]:
# Calculate ghost rate by access route
all_resp = pd.concat([listeners_response, non_listeners_some, non_listeners_empty], ignore_index=True)
ghost_rate = (all_resp.assign(is_ghost=all_resp['id'].isin(non_listeners_empty['id']))
    .groupby('code_category').agg(total=('id', 'count'), ghosts=('is_ghost', 'sum'))
    .assign(ghost_pct=lambda d: (d['ghosts'] / d['total'] * 100).round(1))
    .sort_values('ghost_pct', ascending=False).reset_index())

# Create bar chart of ghost rate by access route
fig = px.bar(ghost_rate, x='code_category', y='ghost_pct', text='ghost_pct',
    title='Ghost Rate by Access Route',
    labels={'code_category': 'Route of access', 'ghost_pct': 'Ghost rate (%)'},
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    marker_color='#646464',
    width=0.5
)

fig.update_layout(
    height=430, width=600,
    showlegend=False,
    margin=dict(t=70, r=40),
    yaxis_range=[0, 80],
    xaxis=dict(
        tickmode='array',
        tickvals=ordered_cats,
        ticktext=[
            f'<span style="color:{"#B8960A" if cat == "PROMOTION" else color_map_referral_codes.get(cat, "#333333")}"><b>{cat}</b></span>'
            for cat in ordered_cats
        ],
        tickangle=-45,
        tickfont=dict(size=12, family='Arial'),
    ),
    plot_bgcolor='#E8EDF4'
)
fig.show()
print(ghost_rate.to_string(index=False))

code_category  total  ghosts  ghost_pct
         FREE  59257   41551       70.1
    PROMOTION    239      37       15.5
      CHARITY    244      26       10.7
    EDUCATION   9302     704        7.6
     EMPLOYER    458      31        6.8
     PURCHASE   2961     173        5.8
          NHS  20606    1129        5.5


## 6.3 Listener Rate & Sustained Engager Rate by Access Route


In [23]:
# Define sustained engagers
sustained_ids = listeners_enriched[sustained_mask]['id']

# Calculate listener and sustained listener rates by access route
listener_rate = (response
    .assign(
        is_listener=response['id'].isin(listeners_response['id']),
        is_sustained=response['id'].isin(sustained_ids)
    )
    .groupby('code_category')
    .agg(total=('id', 'count'), listeners=('is_listener', 'sum'), sustained=('is_sustained', 'sum'))
    .assign(
        listener_pct=lambda d: (d['listeners'] / d['total'] * 100).round(1),
        sustained_pct=lambda d: (d['sustained'] / d['total'] * 100).round(1)
    )
    .sort_values('listener_pct', ascending=False)
    .reset_index())

fig = go.Figure()

# Add bar traces for listeners and sustained listeners
fig.add_trace(go.Bar(
    name='Listeners',
    x=listener_rate['code_category'],
    y=listener_rate['listener_pct'],
    text=listener_rate['listener_pct'].map(lambda v: f'{v:.1f}%'),
    textposition='outside',
    marker_color='#24C955'
))

# Add bar trace for Sustained listeners
fig.add_trace(go.Bar(
    name='Sustained Listeners',
    x=listener_rate['code_category'],
    y=listener_rate['sustained_pct'],
    text=listener_rate['sustained_pct'].map(lambda v: f'{v:.1f}%'),
    textposition='outside',
    marker_color='#115E91'
))

fig.update_layout(
    title_text='Listener & Sustained listeners Rate by Access Route',
    barmode='group',
    height=430, width=750,
    margin=dict(t=70, r=80),
    yaxis=dict(title='Rate (%)', range=[0, 60]),
    xaxis=dict(
        tickmode='array',
        tickvals=ordered_cats,
        ticktext=[
            f'<span style="color:{"#B8960A" if cat == "PROMOTION" else color_map_referral_codes.get(cat, "#333333")}"><b>{cat}</b></span>'
            for cat in ordered_cats
        ],
        tickangle=-45,
        tickfont=dict(size=12, family='Arial'),
    ),
    xaxis_title='Route of access',
    plot_bgcolor='#E8EDF4',
    legend=dict(orientation='v', x=1.02, y=0.5),
    bargap=0.3,
    bargroupgap=0.05
)

fig.show()
print(listener_rate[['code_category', 'total', 'listener_pct', 'sustained_pct']].to_string(index=False))

code_category  total  listener_pct  sustained_pct
          NHS  20606          50.1           19.5
     PURCHASE   2961          45.6           32.4
    EDUCATION   9302          43.3           15.0
     EMPLOYER    458          36.0           13.3
      CHARITY    244          32.0            8.6
    PROMOTION    239          26.4            5.0
         FREE  59257          25.7            4.5


---
# 7. Ghost User Analysis

## 7.1 Time to Ghost

In [24]:
# Identify all unique user IDs from the tracks dataset
all_track_users = set(tracks["id"].dropna().unique())

response_cols = [
    c for c in [
        'q_phq8_0',
        'q_gad7_0',
        'q_psyclops_0',
        'would_use_again2',
        'tracks_listened0',
        'mood'
    ]
    if c in response.columns
]

no_tracks = ~response["id"].isin(all_track_users)
no_meaningful_response = response[response_cols].isnull().all(axis=1)

ghost_df = response[no_tracks & no_meaningful_response].copy()

# Convert user_created_on_time and last_open_time to datetime
ghost_df["user_created_on_time"] = pd.to_datetime(
    ghost_df["user_created_on_time"], format="%y-%m-%d-at-%I-%M%p", errors="coerce"
)
ghost_df["last_open_time"] = pd.to_datetime(
    ghost_df["last_open_time"], format="%y-%m-%d-at-%I-%M%p", errors="coerce"
)

# Calculate days between account creation and last open
ghost_df["dayssincefirstopen"] = (
    ghost_df["last_open_time"] - ghost_df["user_created_on_time"]
).dt.days

ghost_df["dayssincefirstopen"] = ghost_df["dayssincefirstopen"].fillna(0).clip(lower=0)

# Coerce dayssincefirstopen to numeric
ghost_df["dayssincefirstopen"] = pd.to_numeric(
    ghost_df["dayssincefirstopen"], errors="coerce"
)

ghost_days = ghost_df["dayssincefirstopen"].dropna()

bins   = [0, 1, 4, 8, 15, 31, ghost_days.max() + 1]
labels = ["Same day\n(day 0)", "1–3\ndays", "4–7\ndays",
          "8–14\ndays", "15–30\ndays", "30+\ndays"]

ghost_df["days_bin"] = pd.cut(
    ghost_df["dayssincefirstopen"],
    bins=bins, labels=labels, right=False, include_lowest=True
)

# Count the number of ghost users in each bin and calculate percentages
bin_counts = ghost_df["days_bin"].value_counts().reindex(labels).fillna(0)
pct        = (bin_counts / bin_counts.sum() * 100).round(1)

bar_colours = GREY

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=labels,
    y=bin_counts.values,
    marker_color=bar_colours,
    text=[f"{p}%" for p in pct.values],
    textposition="outside",
    textfont=dict(size=12, family="Arial"),
    hovertemplate="<b>%{x}</b><br>Users: %{y}<br>%{text}<extra></extra>",
    name="Ghost users"
))


fig1.update_layout(
    title=dict(
        text="Time to Ghost"
    ),
    height=430, width=750,
    xaxis=dict(title="Days since first open", tickfont=dict(size=12)),
    yaxis=dict(title="Number of ghost users", range=[0, 50000]),
    plot_bgcolor='#E8EDF4',
    showlegend=False,
    margin=dict(t=70, r=80),
    bargap=0.3
)

fig1.show()

---
# 8. Engagement Patterns — All Listeners


## 8.1 Listening Frequency Distribution


In [25]:
# Listening frequency per user
listeners_tracks["trackduration"] = pd.to_numeric(
    listeners_tracks["trackduration"], errors="coerce"
)

print("\nlistening frequency per user:")
freq = (
    listeners_tracks.groupby("id")
    .agg(
        total_sessions=("trackid", "count"),
        unique_tracks=("trackid", "nunique"),
        unique_days=("listening_date", "nunique"),
        avg_duration=("trackduration", "mean")
    )
    .reset_index()
)
print(f"Avg sessions per user:     {freq['total_sessions'].mean():.1f}")
print(f"Median sessions per user:  {freq['total_sessions'].median():.0f}")
print(f"Avg unique tracks:         {freq['unique_tracks'].mean():.1f}")
print(f"Avg unique listening days: {freq['unique_days'].mean():.1f}")
print(f"\nSession count distribution:")
print(freq["total_sessions"].describe().round(1))


listening frequency per user:
Avg sessions per user:     14.8
Median sessions per user:  3
Avg unique tracks:         3.2
Avg unique listening days: 9.4

Session count distribution:
count    31281.0
mean        14.8
std         72.1
min          1.0
25%          1.0
50%          3.0
75%          7.0
max       4351.0
Name: total_sessions, dtype: float64


In [26]:
# Create a histogram of total sessions per user, capped at the 95th percentile
cap = int(freq["total_sessions"].quantile(0.95))
plot_data = freq["total_sessions"].clip(upper=cap)

# set up the histogram
fig = go.Figure(go.Histogram(
    x=plot_data,
    nbinsx=50,
    marker=dict(color="#1D9E75", opacity=0.85, line=dict(color="white", width=0.5)),
    hovertemplate="Sessions: %{x}<br>Users: %{y:,}<extra></extra>",
))

median_val = freq["total_sessions"].median()
mean_val   = freq["total_sessions"].mean().round(1)

# create vertical lines for median and mean
for val, label, colour in [
    (median_val, f"Median: {int(median_val)}", "#378ADD"),
    (mean_val,   f"Mean: {mean_val}",          "#EF9F27"),
]:
    fig.add_vline(
        x=val,
        line_width=2,
        line_dash="dash",
        line_color=colour,
    )
    fig.add_annotation(
        x=val,
        y=1,
        yref="paper",
        text=f"  {label}",
        showarrow=False,
        font=dict(color=colour, size=12),
        xanchor="left",
    )

fig.update_layout(
    title=dict(
        text=f"Distribution of Listening Sessions per User (capped at 95th percentile - {cap} sessions)",
        font=dict(size=15, color="#222")
    ),
    width=750,
    xaxis=dict(title="Total Sessions", showgrid=True, gridcolor="#ececec"),
    yaxis=dict(title="Number of Users", showgrid=True, gridcolor="#ececec"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=40, t=70, b=60),
    height=440,
    bargap=0.05,
)

fig.show()

## 8.2 Listening Activity — Day × Hour Heatmap


In [27]:
# Day × hour heatmap
if 'listening_weekday' in listeners_tracks.columns:
    DAY_ORDER=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot=(listeners_tracks.groupby(['listening_weekday','listening_hour'])['trackid'].count()
           .unstack(fill_value=0).reindex(DAY_ORDER))
    fig2=go.Figure(go.Heatmap(z=pivot.values,
        x=[f'{int(h):02d}' for h in pivot.columns],
        y=pivot.index.tolist(),colorscale='Blues',colorbar=dict(title='Listens')))
    fig2.update_layout(title='Listening Activity — Day × Hour',
        xaxis_title='Hour',height=320,width=750,margin=dict(t=50,l=100))
    fig2.show()


## 8.3 Most Listened Tracks — Volume & Completion Rate


In [28]:
listeners_tracks["progresspercent_clean"] = pd.to_numeric(
    listeners_tracks["progresspercent_clean"], errors="coerce"
)

track_counts = (
    listeners_tracks["trackid"]
    .value_counts()
    .reset_index()
)

# Rename columns for clarity
track_counts.columns = ["trackid", "listen_count"]
track_counts["pct"] = (track_counts["listen_count"] / len(listeners_tracks) * 100).round(1)

# Calculate average completion percentage for each track
completion = (
    listeners_tracks.groupby("trackid")["progresspercent_clean"]
    .mean()
    .round(1)
    .reset_index()
    .rename(columns={"progresspercent_clean": "avg_completion"})
)

# Merge the track counts with the average completion percentages
top15 = (
    track_counts.head(15)
    .merge(completion, on="trackid", how="left")
    .iloc[::-1]
    .reset_index(drop=True)
)

# Create labels for the y-axis by replacing underscores with spaces and capitalizing each word
labels = top15["trackid"].str.replace("_", " ").str.title()

# Create a horizontal bar chart using Plotly
fig = go.Figure(go.Bar(
    x=top15["listen_count"],
    y=labels,
    orientation="h",
    marker=dict(
        color=top15["avg_completion"],
        colorscale=[
            [0.0,  "#D7191C"],  # low completion
            [0.5,  "#EF9F27"],  # mid
            [1.0,  "#1D9E75"],  # high completion
        ],
        cmin=0,
        cmax=100,
        colorbar=dict(
            title=dict(text="Avg Completion (%)", side="right"),
            ticksuffix="%",
            thickness=14,
            len=0.8,
        ),
        line=dict(width=0),
    ),
    text=[f"{p}%  |  {c}% complete" for p, c in zip(top15["pct"], top15["avg_completion"])],
    textposition="outside",
    textfont=dict(size=10, color="#444"),
    hovertemplate="<b>%{y}</b><br>Listens: %{x:,}<br>%{text}<extra></extra>",
))

fig.update_layout(
    title=dict(text="Most Listened Tracks Amongst All Listeners — Volume & Completion Rate",),
    xaxis=dict(
        title="Listen Count",
        showgrid=True, gridcolor="#ececec",
        tickformat=",", zeroline=False,
        range=[0, 105000],
    ),
    yaxis=dict(title=None, tickfont=dict(size=12)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=180, r=100, t=60, b=50),
    height=1000,
    showlegend=False,
)

fig.show()

## 8.4 Gap Profiles


In [29]:
DATE_COL = 'listening_date'

# per user gap calculation
lt = listeners_tracks.copy()
lt[DATE_COL] = pd.to_datetime(lt[DATE_COL], errors='coerce')
lt = lt.dropna(subset=[DATE_COL])

lt_sorted = lt.sort_values(['id', DATE_COL])
lt_sorted['prev_date'] = lt_sorted.groupby('id')[DATE_COL].shift(1)
lt_sorted['gap_days'] = (lt_sorted[DATE_COL] - lt_sorted['prev_date']).dt.days

gaps = lt_sorted.dropna(subset=['gap_days']).copy()

# per user gap summary
user_gap_summary = (
    gaps.groupby('id')['gap_days']
    .agg(median_gap='median', mean_gap='mean', max_gap='max', gap_count='count')
    .round(1)
    .reset_index()
)

# gap profile classification
def gap_type(median):
    if median <= 2:   return 'Habitual (≤2 days)'
    elif median <= 7:  return 'Regular (3–7 days)'
    elif median <= 30: return 'Intermittent (8–30 days)'
    else:              return 'Lapsing (>30 days)'

user_gap_summary['gap_profile'] = user_gap_summary['median_gap'].apply(gap_type)

# setting up one histogram to show Gap distribution
gap_plot = gaps[gaps['gap_days'] <= 90]['gap_days']
med = gap_plot.median()
mn  = gap_plot.mean()

fig1 = go.Figure()

fig1.add_trace(go.Histogram(
    x=gap_plot,
    nbinsx=60,
    marker_color='#378ADD',
    marker_line=dict(width=0.4, color='white'),
    name='Gap frequency',
    hovertemplate='Gap: %{x} days<br>Count: %{y}<extra></extra>'
))
fig1.add_vline(x=med, line=dict(color='#1D9E75', width=2, dash='dash'),
               annotation_text=f'Median: {med:.0f}d',
               annotation_position='top right',
               annotation_font=dict(color='#1D9E75', size=11))
fig1.add_vline(x=mn, line=dict(color='#EF9F27', width=2, dash='dash'),
               annotation_text=f'Mean: {mn:.1f}d',
               annotation_position='top left',
               annotation_font=dict(color='#EF9F27', size=11))
fig1.update_layout(
    title=dict(text='Gap Distribution Between Sessions (capped at 90 days)',
               font=dict(size=16, color='#222'), x=0),
    xaxis=dict(title='Days Between Consecutive Listens', showgrid=True,
               gridcolor='#ececec', zeroline=False),
    yaxis=dict(title='Frequency', showgrid=True, gridcolor='#ececec'),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=60, b=50, l=60, r=40),
    height=380, showlegend=False
)
fig1.show()

# setting up a pie chart to show Gap profile distribution
profile_counts = user_gap_summary['gap_profile'].value_counts().reset_index()
profile_counts.columns = ['gap_profile', 'count']

PROFILE_COLOURS = {
    'Habitual (≤2 days)':      '#1D9E75',
    'Regular (3–7 days)':       '#378ADD',
    'Intermittent (8–30 days)': '#EF9F27',
    'Lapsing (>30 days)':       '#888780',
}
colours = [PROFILE_COLOURS.get(p, '#cccccc') for p in profile_counts['gap_profile']]

fig2 = go.Figure(go.Pie(
    labels=profile_counts['gap_profile'],
    values=profile_counts['count'],
    hole=0.5,
    marker=dict(colors=colours),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Users: %{value:,}<br>%{percent}<extra></extra>'
))
fig2.update_layout(
    title=dict(text='Users by Gap Profile (Median Inter-Session Gap)',
               font=dict(size=16, color='#222'), x=0),
    paper_bgcolor='white',
    margin=dict(t=60, b=40, l=40, r=40),
    height=420, showlegend=True,
    legend=dict(orientation='v', x=1.02, y=0.5)
)
fig2.show()

In [30]:
# Flag single-session users
session_counts = lt.groupby('id')['listening_date'].count().reset_index()
session_counts.columns = ['id', 'session_count']

single_session_ids = session_counts[session_counts['session_count'] == 1]['id']

# Add them as a category
single_df = pd.DataFrame({
    'id': single_session_ids,
    'gap_profile': 'Single Session'
})

user_gap_full = pd.concat([
    user_gap_summary[['id', 'gap_profile']],
    single_df
], ignore_index=True)

# Setting up Donut chart for listening consistency
PROFILE_ORDER = [
    'Habitual (≤2 days)',
    'Regular (3–7 days)',
    'Intermittent (8–30 days)',
    'Lapsing (>30 days)',
    'Single Session',
]

PROFILE_COLOURS = {
    'Habitual (≤2 days)':      '#0D5C2E',  
    'Regular (3–7 days)':       '#24C955',  
    'Intermittent (8–30 days)': '#7DDBA0', 
    'Lapsing (>30 days)':       '#B8D9C3', 
    'Single Session':           '#E0E0E0',  
}

profile_counts = (
    user_gap_full['gap_profile']
    .value_counts()
    .reindex(PROFILE_ORDER)
    .dropna()
    .reset_index()
)

profile_counts.columns = ['gap_profile', 'count']
colours = [PROFILE_COLOURS[p] for p in profile_counts['gap_profile']]

fig = go.Figure(go.Pie(
    labels=profile_counts['gap_profile'],
    values=profile_counts['count'],
    hole=0.5,
    marker=dict(colors=colours),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Users: %{value:,}<br>%{percent}<extra></extra>'
))

fig.update_traces(
    sort=False,
    direction='clockwise',
    rotation=90,
    marker=dict(
        colors=colours,
        line=dict(color='white', width=2)
    )
)

fig.update_layout(
    title=dict(
        text='Listening Consistency Among Listeners',
        x=0.03,
        font=dict(size=18, family='Arial')
    ),
    height=450,
    width=750,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=60, b=30, l=30, r=180),
    legend=dict(
        title='Gap profile',
        orientation='v',
        x=1.02,
        y=0.5
    ),
    font=dict(family='Arial', size=12, color='#3D3D3D')
)

fig.show()

In [31]:
# gap profile by route of access

user_gap_route = user_gap_full.merge(
    listeners_enriched[['id', 'code_category']].drop_duplicates('id'),
    on='id', how='left'
)

# Cross-tab: proportions within each route
crosstab = (
    user_gap_route.groupby(['code_category', 'gap_profile'])
    .size()
    .reset_index(name='count')
)
crosstab['pct'] = (
    crosstab.groupby('code_category')['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)


# set up Stacked bar chart to showing gap profile % by route
PROFILE_ORDER = [
    'Habitual (≤2 days)',
    'Regular (3–7 days)',
    'Intermittent (8–30 days)',
    'Lapsing (>30 days)',
    'Single Session',
]
PROFILE_COLOURS = {
    'Habitual (≤2 days)':      '#0D5C2E',  # deep forest green
    'Regular (3–7 days)':       '#24C955',  # bright green
    'Intermittent (8–30 days)': '#7DDBA0',  # light green
    'Lapsing (>30 days)':       '#B8D9C3',  # muted sage
    'Single Session':           '#E0E0E0',  # neutral gray
}

# Route order by descending sustained %
sustained = crosstab[crosstab['gap_profile'] == 'Habitual (≤2 days)']
route_order = sustained.sort_values('pct', ascending=False)['code_category'].tolist()

fig = go.Figure()

for profile in PROFILE_ORDER:
    subset = crosstab[crosstab['gap_profile'] == profile].set_index('code_category')
    fig.add_trace(go.Bar(
        name=profile,
        x=route_order,
        y=[subset.loc[r, 'pct'] if r in subset.index else 0 for r in route_order],
        marker_color=PROFILE_COLOURS[profile],
        text=[f"{subset.loc[r, 'pct']:.1f}%" if r in subset.index else '' for r in route_order],
        textposition='inside',
        textfont=dict(color='white', size=11),
        hovertemplate='<b>%{x}</b><br>' + profile + ': %{y:.1f}%<extra></extra>'
    ))

fig.update_layout(
    barmode='stack',
    title=dict(text='Listening Consistency of All Listeners by Route of Access',
               ),
    xaxis=dict(
        title='Route of Access',
        tickmode='array',
        tickvals=ordered_cats,
        ticktext=[
            f'<span style="color:{"#B8960A" if cat == "PROMOTION" else color_map_referral_codes.get(cat, "#333333")}"><b>{cat}</b></span>'
            for cat in ordered_cats
        ],
        tickfont=dict(size=12, family='Arial'),
    ),
    yaxis=dict(title='% of Users', range=[0, 100],
               ticksuffix='%', showgrid=True, gridcolor='#ececec'),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Gap Profile', orientation='v', x=1.02, y=0.5),
    margin=dict(t=60, b=50, l=60, r=180),
    height=440
)
fig.show()

## 8.5 User Tenure by Access Route


In [32]:
# tenure analysis

# Calculate the tenure for each user based on their first and last listening dates.
tenure = (
    lt.groupby('id')[DATE_COL]
    .agg(first_listen='min', last_listen='max')
    .reset_index()
)
tenure['tenure_days'] = (tenure['last_listen'] - tenure['first_listen']).dt.days

# Merge the tenure data with the code_category information from listeners_enriched.
tenure = tenure.merge(
    listeners_enriched[['id', 'code_category']].drop_duplicates('id'),
    on='id', how='left'
)

# Summarize the tenure statistics by code_category, including count, median, mean, and percentage of users with tenure over 30 and 90 days.
tenure_summary = (
    tenure.groupby('code_category')['tenure_days']
    .agg(
        n='count',
        median_tenure='median',
        mean_tenure='mean',
        pct_over_30=lambda x: (x >= 30).mean() * 100,
        pct_over_90=lambda x: (x >= 90).mean() * 100
    )
    .round(1)
    .reset_index()
    .sort_values('median_tenure', ascending=False)
)

print(tenure_summary.to_string(index=False))


cat_order = tenure_summary['code_category'].tolist()

fig3 = go.Figure()

for cat in cat_order:
    data = tenure[tenure['code_category'] == cat]['tenure_days'].dropna()
    fig3.add_trace(go.Box(
        y=data,
        name=cat,
        marker_color=color_map_referral_codes.get(cat, "#0CB946"),
        line=dict(width=1.5),
        boxmean=True,  # shows mean as dashed line
        hovertemplate=f'<b>{cat}</b><br>Tenure: %{{y}} days<extra></extra>'
    ))



fig3 = go.Figure()


for cat in cat_order:
    df_cat = tenure[tenure['code_category'] == cat]['tenure_days'].dropna()
    color = color_map_referral_codes.get(cat, '#888780')

    fig3.add_trace(go.Box(
        y=df_cat,
        name=cat,
        marker_color=color,
        line=dict(color=color, width=1.2),
        fillcolor=color,
        opacity=0.6,
        boxmean=True,
        whiskerwidth=0.6,
        marker=dict(size=3, opacity=0.4),
        hovertemplate=(
            f"<b>{cat}</b><br>"
            "Tenure: %{y} days<extra></extra>"
        )
    ))

    fig3.add_trace(go.Box(
        y=df_cat,
        name=cat,
        marker=dict(color=color, opacity=0.25, size=4),
        boxpoints='all',
        jitter=0.35,
        pointpos=0,
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))


fig3.update_layout(
    title=dict(
        text='User Tenure by Route of Access'
    ),
    xaxis=dict(
        title='Route of Access',
        categoryorder='array',
        categoryarray=cat_order,
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title='Tenure (days)',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.06)',
        zeroline=False
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=70, b=60, l=70, r=40),
    height=520,
    width=700,
    showlegend=False
)


medians = (
    tenure.groupby('code_category')['tenure_days']
    .median()
    .reindex(cat_order)
)


fig3.add_trace(go.Scatter(
    x=cat_order,
    y=medians,
    mode='markers+text',
    text=[f"{v:.0f}" for v in medians],
    textposition="top center",
    marker=dict(size=10, color="black", symbol="diamond"),
    name="Median",
    showlegend=False
))

fig3.show()

code_category     n  median_tenure  mean_tenure  pct_over_30  pct_over_90
     PURCHASE  1351          194.0        395.8         71.1         60.5
          NHS 10328           10.0        133.0         39.2         27.6
     EMPLOYER   164            9.5        159.7         37.8         29.3
    EDUCATION  4029            4.0        123.0         34.9         25.7
      CHARITY    69            3.0        124.1         27.5         21.7
    PROMOTION    63            2.0         69.3         20.6         12.7
         FREE 15201            0.0         52.6         17.5         12.0


---
# 9. Sustained Engager Analysis


## 9.1 Deep Dive — Route, Tenure & Clinical Change


In [33]:
# building sustained engagers dataset
sustained_users = listeners_enriched[sustained_mask].copy()

# Coerce clinical columns
for col in ["listening_span_days", "q_gad7_0", "q_gad7_2", "q_gad7_4",
            "q_phq8_0", "q_phq8_2", "q_phq8_4"]:
    sustained_users[col] = pd.to_numeric(sustained_users[col], errors="coerce")

# Change scores
sustained_users["GAD7_change"] = sustained_users["q_gad7_2"].fillna(sustained_users["q_gad7_4"]) - sustained_users["q_gad7_0"]
sustained_users["PHQ8_change"] = sustained_users["q_phq8_2"].fillna(sustained_users["q_phq8_4"]) - sustained_users["q_phq8_0"]

# Create a bar chart for code categories among sustained engagers
cat_counts = sustained_users["code_category"].value_counts().reset_index()
cat_counts.columns = ["code_category", "count"]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=cat_counts["code_category"],
    y=cat_counts["count"],
    marker_color=[
        color_map_referral_codes.get(c, "#aaaaaa")
        for c in cat_counts["code_category"]
    ],
    text=cat_counts["count"],
    textposition="outside",
    hovertemplate="%{x}<br>Users: %{y}<extra></extra>",
    showlegend=False,
))

fig.update_layout(
    title=dict(
        text=f"Sustained Engager Deep Dive  (30+ day span, 2+ active weeks, n={len(sustained_users):,})",
        font=dict(size=16, color="#222"),
        x=0,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=480,
    margin=dict(l=50, r=40, t=90, b=60),
    xaxis_title="Code category",
    yaxis_title="Users",
)

fig.update_yaxes(showgrid=True, gridcolor="#ececec")
fig.update_xaxes(showgrid=False)

fig.show()


In [34]:
# Listening span distribution among sustained engagers

span_vals = sustained_users["listening_span_days"].dropna()
span_median = span_vals.median()

fig_span = go.Figure()

# Add histogram trace for listening span
fig_span.add_trace(go.Histogram(
    x=span_vals,
    nbinsx=40,
    marker=dict(
        color=BLUE,
        opacity=0.85,
        line=dict(color="white", width=0.5)
    ),
    hovertemplate="Listening span: %{x:.0f} days<br>Users: %{y}<extra></extra>",
    showlegend=False
))

# Add median line
fig_span.add_vline(
    x=span_median,
    line_dash="dash",
    line_color=AMBER,
    line_width=2,
    annotation_text=f"Median: {int(span_median)} days",
    annotation_position="top right",
    annotation_font=dict(color=AMBER, size=11)
)

fig_span.update_layout(
    title=dict(
        text="Listening Span Among Sustained Engagers",
        x=0.03,
        font=dict(size=18, family="Arial")
    ),
    xaxis=dict(title="Listening span days"),
    yaxis=dict(title="Number of sustained engagers"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=420,
    width=700,
    margin=dict(l=70, r=40, t=80, b=70),
    font=dict(family="Arial", size=12, color="#3D3D3D")
)

fig_span.update_yaxes(gridcolor="#EEEEEE", zeroline=False)
fig_span.update_xaxes(showgrid=False)

fig_span.show()

In [35]:
# Mean clinical change among sustained engagers

sustained_users = listeners_enriched[sustained_mask].copy()
measures = ["GAD-7", "PHQ-8"]

means = [
    sustained_users["gad7_change"].mean(),
    sustained_users["phq8_change"].mean()
]

ns = [
    sustained_users["gad7_change"].notna().sum(),
    sustained_users["phq8_change"].notna().sum()
]

fig_change = go.Figure()

# Add bar traces for each measure
fig_change.add_trace(go.Bar(
    x=measures,
    y=means,
    marker_color=[TEAL if v < 0 else RED for v in means],
    text=[f"{v:.2f}<br>n={n}" for v, n in zip(means, ns)],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Mean change: %{y:.2f}<extra></extra>",
    showlegend=False
))

# Add a horizontal line at y=0
fig_change.add_hline(
    y=0,
    line_color=GREY,
    line_width=1.2
)

fig_change.update_layout(
    title=dict(
        text="Mean Clinical Change Among Sustained Engagers",
        x=0.03,
        font=dict(size=18, family="Arial")
    ),
    xaxis=dict(title="Outcome measure"),
    yaxis=dict(title="Mean change score"),
    yaxis_range=[min(means) - 1, 1],
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=420,
    width=700,
    margin=dict(l=70, r=40, t=80, b=70),
    font=dict(family="Arial", size=12, color="#3D3D3D")
)

fig_change.update_yaxes(gridcolor="#EEEEEE", zeroline=False)

fig_change.show()

## 9.2 Listening Patterns — Day × Hour Heatmap


In [36]:
# Filter tracks to sustained users only
sustained_tracks = listeners_tracks[listeners_tracks['id'].isin(sustained_users['id'])]

# Day × hour heatmap — sustained engagers
if 'listening_weekday' in sustained_tracks.columns:
    DAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = (sustained_tracks.groupby(['listening_weekday', 'listening_hour'])['trackid'].count()
             .unstack(fill_value=0).reindex(DAY_ORDER))
    fig2 = go.Figure(go.Heatmap(
        z=pivot.values,
        x=[f'{int(h):02d}:00' for h in pivot.columns],
        y=pivot.index.tolist(),
        colorscale='Blues',
        colorbar=dict(title='Listens')))
    fig2.update_layout(
        title=f'Listening Activity — Day × Hour (Sustained Engagers, n={len(sustained_users):,})',
        xaxis_title='Hour',
        height=320,
        width=750,
        margin=dict(t=50, l=100))
    fig2.show()




## 9.3 Most Listened Tracks


In [37]:
#sustained users top tracks by listen count and average completion
lt_sustained = listeners_tracks[listeners_tracks['id'].isin(sustained_users['id'])].copy()

lt_sustained["progresspercent_clean"] = pd.to_numeric(
    lt_sustained["progresspercent_clean"], errors="coerce"
)

# top tracks by listen count and average completion
track_counts = (
    lt_sustained["trackid"]
    .value_counts()
    .reset_index()
)
track_counts.columns = ["trackid", "listen_count"]
track_counts["pct"] = (track_counts["listen_count"] / len(lt_sustained) * 100).round(1)

completion = (
    lt_sustained.groupby("trackid")["progresspercent_clean"]
    .mean()
    .round(1)
    .reset_index()
    .rename(columns={"progresspercent_clean": "avg_completion"})
)

top15 = (
    track_counts.head(15)
    .merge(completion, on="trackid", how="left")
    .iloc[::-1]
    .reset_index(drop=True)
)

labels = top15["trackid"].str.replace("_", " ").str.title()

# Create a horizontal bar chart with color representing average completion
fig = go.Figure(go.Bar(
    x=top15["listen_count"],
    y=labels,
    orientation="h",
    marker=dict(
        color=top15["avg_completion"],
        colorscale=[
            [0.0, "#D7191C"],
            [0.5, "#7FB3D5"],
            [1.0, "#0B3C5D"],
        ],
        cmin=0,
        cmax=100,
        colorbar=dict(
            title=dict(text="Avg Completion (%)", side="right"),
            ticksuffix="%",
            thickness=14,
            len=0.8,
        ),
        line=dict(width=0),
    ),
    text=[f"{p}%  |  {c}% complete" for p, c in zip(top15["pct"], top15["avg_completion"])],
    textposition="outside",
    textfont=dict(size=10, color="#444"),
    hovertemplate="<b>%{y}</b><br>Listens: %{x:,}<br>%{text}<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=f"Most Listened Tracks — Sustained Engagers (30+ day span, 2+ active weeks, n={len(sustained_users):,})",
    ),
    xaxis=dict(
        title="Listen Count",
        showgrid=True, gridcolor="#ececec",
        tickformat=",", zeroline=False,
        range=[0, lt_sustained['trackid'].value_counts().iloc[0] * 1.25],
    ),
    yaxis=dict(title=None, tickfont=dict(size=12)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=180, r=100, t=60, b=50),
    height=1000,
    showlegend=False,
)

fig.show()

## 9.4 Gap Profile Distribution


In [38]:
#gap profile for sustained engagers
lt_sustained = lt[lt['id'].isin(sustained_users['id'])].copy()

# gap calculation for sustained engagers
lt_sustained_sorted = lt_sustained.sort_values(['id', DATE_COL])
lt_sustained_sorted['prev_date'] = lt_sustained_sorted.groupby('id')[DATE_COL].shift(1)
lt_sustained_sorted['gap_days'] = (lt_sustained_sorted[DATE_COL] - lt_sustained_sorted['prev_date']).dt.days

gaps_sustained = lt_sustained_sorted.dropna(subset=['gap_days']).copy()

user_gap_summary_sustained = (
    gaps_sustained.groupby('id')['gap_days']
    .agg(median_gap='median', mean_gap='mean', max_gap='max', gap_count='count')
    .round(1)
    .reset_index()
)
user_gap_summary_sustained['gap_profile'] = user_gap_summary_sustained['median_gap'].apply(gap_type)

user_gap_full_sustained = user_gap_summary_sustained[['id', 'gap_profile']].copy()

# donut chart for gap profile of sustained engagers
PROFILE_ORDER = [
    'Habitual (≤2 days)',
    'Regular (3–7 days)',
    'Intermittent (8–30 days)',
    'Lapsing (>30 days)',
]
PROFILE_COLOURS = {
    'Habitual (≤2 days)':      '#0B3C5D',
    'Regular (3–7 days)':       '#328CC1',
    'Intermittent (8–30 days)': '#7FB3D5',
    'Lapsing (>30 days)':       '#B8C7D1',
}

profile_counts_sustained = (
    user_gap_full_sustained['gap_profile']
    .value_counts()
    .reindex(PROFILE_ORDER)
    .dropna()
    .reset_index()
)
profile_counts_sustained.columns = ['gap_profile', 'count']
donut_colours = [PROFILE_COLOURS[p] for p in profile_counts_sustained['gap_profile']]

fig = go.Figure(go.Pie(
    labels=profile_counts_sustained['gap_profile'],
    values=profile_counts_sustained['count'],
    hole=0.5,
    marker=dict(colors=donut_colours, line=dict(color='white', width=2)),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Users: %{value:,}<br>%{percent}<extra></extra>'
))
fig.update_traces(sort=False, direction='clockwise', rotation=90,
                  insidetextorientation='horizontal')
fig.update_layout(
    title=dict(text=f'Gap Profile — Sustained Engagers (n={len(sustained_users):,})'),
    paper_bgcolor='white', margin=dict(t=60, b=40, l=40, r=40),
    height=420, width=700, showlegend=True, legend=dict(orientation='v', x=1.02, y=0.5)
)
fig.show()


## 9.5 Gap Profile × Access Route


In [39]:
# gap profile by route of access for sustained engagers
user_gap_route_sustained = user_gap_full_sustained.merge(
    listeners_enriched[['id', 'code_category']].drop_duplicates('id'),
    on='id', how='left'
)

# create crosstab of gap profile by route of access
crosstab_sustained = (
    user_gap_route_sustained.groupby(['code_category', 'gap_profile'])
    .size()
    .reset_index(name='count')
)
crosstab_sustained['pct'] = (
    crosstab_sustained.groupby('code_category')['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

# set up chart
PROFILE_ORDER = [
    'Habitual (≤2 days)',
    'Regular (3–7 days)',
    'Intermittent (8–30 days)',
    'Lapsing (>30 days)',
    'Single Session',
]
PROFILE_COLOURS = {
    'Habitual (≤2 days)':      '#0B3C5D',
    'Regular (3–7 days)':       '#328CC1',
    'Intermittent (8–30 days)': '#7FB3D5',
    'Lapsing (>30 days)':       '#B8C7D1',
    'Single Session':           '#E0E0E0',
}

# filter to sustained engagers only for route order
sustained_profile = crosstab_sustained[crosstab_sustained['gap_profile'] == 'Habitual (≤2 days)']
route_order_sustained = sustained_profile.sort_values('pct', ascending=False)['code_category'].tolist()

fig = go.Figure()

# build stacked bar chart for gap profile by route of access
for profile in PROFILE_ORDER:
    subset = crosstab_sustained[crosstab_sustained['gap_profile'] == profile].set_index('code_category')
    fig.add_trace(go.Bar(
        name=profile,
        x=route_order_sustained,
        y=[subset.loc[r, 'pct'] if r in subset.index else 0 for r in route_order_sustained],
        marker_color=PROFILE_COLOURS[profile],
        text=[f"{subset.loc[r, 'pct']:.1f}%" if r in subset.index else '' for r in route_order_sustained],
        textposition='inside',
        textfont=dict(color='white' if profile != 'Single Session' else '#666', size=11),
        hovertemplate='<b>%{x}</b><br>' + profile + ': %{y:.1f}%<extra></extra>'
    ))

fig.update_layout(
    barmode='stack',
    title=dict(
        text=f'Listening Consistency Among Sustained Engagers, by Access Route (n={len(sustained_users):,})'
    ),
    xaxis=dict(
        title='Route of Access',
        tickmode='array',
        tickvals=ordered_cats,
        ticktext=[
            f'<span style="color:{"#B8960A" if cat == "PROMOTION" else color_map_referral_codes.get(cat, "#333333")}"><b>{cat}</b></span>'
            for cat in ordered_cats
        ],
        tickfont=dict(size=12, family='Arial'),
    ),
    yaxis=dict(title='% of Users', range=[0, 100],
               ticksuffix='%', showgrid=True, gridcolor='#ececec'),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Gap Profile', orientation='v', x=1.02, y=0.5),
    margin=dict(t=60, b=50, l=60, r=180),
    height=440
)
fig.show()

---
# 10. Clinical Outcomes


## 10.1 Clinical Change Score Distributions



In [40]:
# clinical change distribution among sustained engagers

outcome_cols = [
    ('gad7_change', 'GAD-7', BLUE, -5),
    ('phq8_change', 'PHQ-8', TEAL, -5)
]

fig_dist = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[label for _, label, _, _ in outcome_cols],
    horizontal_spacing=0.12
)

for col_num, (change_col, label, colour, responder_threshold) in enumerate(outcome_cols, start=1):
    vals = listeners_enriched[change_col].dropna()
    n = len(vals)
    if n == 0:
        continue

    responders = (vals <= responder_threshold).sum()
    deteriorated = (vals > 0).sum()

    # Clinically meaningful improvement zone: −5 or lower.
    fig_dist.add_vrect(
        x0=vals.min() - 1,
        x1=responder_threshold,
        fillcolor=TEAL,
        opacity=0.07,
        line_width=0,
        row=1,
        col=col_num
    )

    # Deterioration zone: above zero.
    fig_dist.add_vrect(
        x0=0,
        x1=vals.max() + 1,
        fillcolor=RED,
        opacity=0.07,
        line_width=0,
        row=1,
        col=col_num
    )

    fig_dist.add_trace(
        go.Histogram(
            x=vals,
            nbinsx=30,
            marker_color=colour,
            opacity=0.85,
            hovertemplate='Change score: %{x}<br>Count: %{y}<extra></extra>',
            showlegend=False
        ),
        row=1,
        col=col_num
    )

    fig_dist.add_vline(x=0, line_dash='dot', line_color=GREY, line_width=1.5, row=1, col=col_num)
    fig_dist.add_vline(x=responder_threshold, line_dash='dash', line_color=TEAL, line_width=2, row=1, col=col_num)

    y_max = np.histogram(vals, bins=30)[0].max()

    fig_dist.add_annotation(
        x=responder_threshold,
        y=y_max * 1.05,
        text='Responder<br>threshold',
        showarrow=False,
        font=dict(size=9, color=TEAL),
        xanchor='right',
        row=1,
        col=col_num
    )

    fig_dist.add_annotation(
        x=vals.min(),
        y=y_max * 0.92,
        text=f'Improved<br>n={responders} ({responders/n*100:.0f}%)',
        showarrow=False,
        font=dict(size=10, color=TEAL),
        xanchor='left',
        bgcolor='rgba(255,255,255,0.8)',
        row=1,
        col=col_num
    )

    fig_dist.add_annotation(
        x=vals.max(),
        y=y_max * 0.92,
        text=f'Worsened<br>n={deteriorated} ({deteriorated/n*100:.0f}%)',
        showarrow=False,
        font=dict(size=10, color=RED),
        xanchor='right',
        bgcolor='rgba(255,255,255,0.8)',
        row=1,
        col=col_num
    )

fig_dist.update_layout(
    title=dict(
        text=(
            '<b>Distribution of Clinical Change Scores</b><br>'
            '<span style="font-size:13px;color:#888780">'
            'Negative = improvement · Dashed line = −5-point responder threshold</span>'
        ),
        x=0.03,
        font=dict(size=18, family='Arial')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=120, b=60, l=60, r=40),
    height=430
)
fig_dist.update_xaxes(title_text='Change score (follow-up − baseline)', gridcolor='#EEEEEE', zeroline=False)
fig_dist.update_yaxes(title_text='Number of users', gridcolor='#EEEEEE', zeroline=False)

fig_dist.show()


## 10.2 Mean Clinical Change by Access Route


In [41]:
# mean clinical change by route of access among sustained engagers

change_cols = {
    'gad7_change': 'GAD-7 Change',
    'phq8_change': 'PHQ-8 Change'
}

fig_route = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=list(change_cols.values()),
    horizontal_spacing=0.12
)

for i, (col, label) in enumerate(change_cols.items(), start=1):
    summary = (
        listeners_enriched.groupby('code_category')[col]
        .agg(['mean', 'sem', 'count'])
        .reset_index()
        .dropna(subset=['mean'])
        .rename(columns={'count': 'n'})
    )

    # Sort by mean change: most negative values indicate greatest improvement.
    summary = summary[summary['n'] >= 5].sort_values('mean', ascending=True)
    bar_colours = [color_map_referral_codes.get(cat, GREY) for cat in summary['code_category']]

    fig_route.add_trace(
        go.Bar(
            x=summary['code_category'],
            y=summary['mean'],
            marker_color=bar_colours,
            error_y=dict(
                type='data',
                array=summary['sem'].values,
                color='#444444',
                thickness=1.5,
                width=5
            ),
            text=[f"n={n}" for n in summary['n']],
            textposition='outside',
            textfont=dict(size=9, color=GREY),
            customdata=summary[['mean', 'sem', 'n']].values,
            hovertemplate=(
                '<b>%{x}</b><br>'
                f'{label}<br>'
                'Mean change: %{customdata[0]:.2f}<br>'
                'SE: ±%{customdata[1]:.2f}<br>'
                'n=%{customdata[2]:.0f}<extra></extra>'
            ),
            showlegend=False
        ),
        row=1,
        col=i
    )

    # Add horizontal lines for no change and the −5 responder threshold.
    fig_route.add_hline(y=0, line_dash='dot', line_color=GREY, line_width=1.2, row=1, col=i)
    fig_route.add_hline(
        y=-5,
        line_dash='dash',
        line_color=TEAL,
        line_width=1.5,
        annotation_text='Responder threshold −5' if i == 1 else '',
        annotation_font=dict(size=9, color=TEAL),
        annotation_position='left',
        row=1,
        col=i
    )

fig_route.update_layout(
    title=dict(
        text=(
            '<b>Mean Clinical Change by Access Route</b><br>'
            '<span style="font-size:13px;color:#888780">'
            'Negative = improvement · Dashed = −5-point responder threshold · n = paired users</span>'
        ),
        x=0.03,
        font=dict(size=18, family='Arial')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=120, b=100, l=70, r=40),
    height=460
)
fig_route.update_xaxes(tickangle=-35, tickfont=dict(size=10), showgrid=False, linecolor='#DDDDDD')
fig_route.update_yaxes(gridcolor='#EEEEEE', zeroline=False, title_text='Mean change score')

fig_route.show()


## 10.3 Responder Rates by Referral Route — All Listeners

In [42]:
# Responder rates by access route among sustained engagers
def plot_responder_rates_negative(
    df,
    title='Responder Rates by Access Route',
    subtitle=None,
    min_n=5
):
    specs = [
        ('gad7_change', 'gad7_responder', 'q_gad7_0', 'GAD-7'),
        ('phq8_change', 'phq8_responder', 'q_phq8_0', 'PHQ-8')
    ]

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[f'{label} Responders' for *_, label in specs],
        horizontal_spacing=0.12
    )

    for i, (change_col, resp_col, base_col, label) in enumerate(specs, start=1):
        if change_col not in df.columns or resp_col not in df.columns:
            print(f"Skipping {label}: missing {change_col} or {resp_col}")
            continue

        paired = df.loc[df[change_col].notna()].copy()
        if paired.empty:
            continue

        summary = (
            paired.groupby('code_category')
            .agg(
                n_paired=(resp_col, 'count'),
                n_responders=(resp_col, 'sum'),
                mean_baseline=(base_col, 'mean'),
                mean_change=(change_col, 'mean')
            )
            .reset_index()
        )

        summary['responder_rate'] = (
            summary['n_responders'] / summary['n_paired'] * 100
        ).round(1)

        summary = (
            summary[summary['n_paired'] >= min_n]
            .sort_values('responder_rate')
        )

        fig.add_trace(
            go.Bar(
                x=summary['responder_rate'],
                y=summary['code_category'],
                orientation='h',
                marker_color=summary['code_category'].map(color_map_referral_codes).fillna(GREY),
                text=[
                    f'{rate:.0f}% (n={n})'
                    for rate, n in zip(summary['responder_rate'], summary['n_paired'])
                ],
                textposition='outside',
                textfont=dict(size=10),
                customdata=summary[
                    ['mean_baseline', 'mean_change', 'n_responders', 'n_paired']
                ].values,
                hovertemplate=(
                    '<b>%{y}</b><br>'
                    'Responder rate: %{x:.1f}%<br>'
                    'Responders: %{customdata[2]:.0f} / %{customdata[3]:.0f}<br>'
                    f'Mean {label} baseline: %{{customdata[0]:.1f}}<br>'
                    'Mean change: %{customdata[1]:.2f}<extra></extra>'
                ),
                showlegend=False
            ),
            row=1,
            col=i
        )

        overall_rate = paired[resp_col].sum() / paired[resp_col].count() * 100
        fig.add_vline(
            x=overall_rate,
            line_dash='dot',
            line_width=1.5,
            annotation_text=f'Overall: {overall_rate:.0f}%',
            annotation_position='top',
            row=1,
            col=i
        )

    subtitle = subtitle or '% of users with paired data showing a change score of −5 or lower'
    fig.update_layout(
    title=dict(
        text=f"<b>{title}</b><br><span style='font-size:13px;color:#888780'>{subtitle}</span>",
        x=0.03,
        font=dict(size=18, family='Arial')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#3D3D3D'),
    height=430,
    margin=dict(t=120, b=60, l=120, r=100),
    showlegend=False
)
    fig.update_xaxes(
        title_text='Responder rate (%)',
        ticksuffix='%',
        range=[0, 110],
        gridcolor='#EEEEEE',
        zeroline=False
    )
    fig.update_yaxes(showgrid=False, tickfont=dict(size=11))
    return fig


plot_responder_rates_negative(listeners_enriched,
   title='Responder Rates by Access Route'
).show()


## 10.4 Responder Rates by Access Route — Sustained Users


In [43]:
sustained_users = listeners_enriched[sustained_mask].copy()

plot_responder_rates_negative(
    sustained_users,
    title='Responder Rates by Access Route — Sustained Users',
    subtitle=f'% of sustained users with paired data showing a change score of −5 or lower (n={len(sustained_users):,})'
).show()


## 10.5 Dose-Response Analysis


In [44]:
# Load data
for measure in ['gad7', 'phq8']:
    for tp in [2, 4, 6, 8]:
        col_name = f'{measure}_change_T{tp}'
        col0 = f'q_{measure}_0'
        colT = f'q_{measure}_{tp}'
        if col_name not in listeners_enriched.columns and colT in listeners_enriched.columns:
            listeners_enriched[col_name] = (
                pd.to_numeric(listeners_enriched[colT], errors='coerce') -
                pd.to_numeric(listeners_enriched[col0], errors='coerce')
            )

EXCLUDE = ['INTERNAL', 'RESEARCH', 'MARKETTING']
df = listeners_enriched[~listeners_enriched['code_category'].isin(EXCLUDE)].copy()

#Bin total_tracks into dose groups for Panel C
df['dose_group'] = pd.cut(
    df['total_tracks'],
    bins=[0, 5, 15, 30, 50, np.inf],
    labels=['1–5', '6–15', '16–30', '31–50', '50+']
)

dose_means = df.groupby('dose_group', observed=True).agg(
    n=('id', 'count'),
    gad7_change=('gad7_change_T2', 'mean'),
    phq8_change=('phq8_change_T2', 'mean'),
    mean_completion=('mean_completion', 'mean'),
).reset_index()

# set up Regression stats helper
def reg_stats(x, y):
    mask = x.notna() & y.notna()
    x_, y_ = x[mask], y[mask]
    slope, intercept, r, p, se = stats.linregress(x_, y_)
    n = mask.sum()
    r2 = r ** 2
    return slope, intercept, r, r2, p, n, x_, y_

# ── Panel A: total_tracks vs GAD-7 change
sA, iA, rA, r2A, pA, nA, xA, yA = reg_stats(df['total_tracks'], df['gad7_change_T2'])
x_lineA = np.linspace(xA.min(), xA.max(), 200)
y_lineA = sA * x_lineA + iA

# ── Panel B: mean_completion vs GAD-7 change
sB, iB, rB, r2B, pB, nB, xB, yB = reg_stats(df['mean_completion'], df['gad7_change_T2'])
x_lineB = np.linspace(xB.min(), xB.max(), 200)
y_lineB = sB * x_lineB + iB

# ── Panel D: total_tracks vs PHQ-8 change
sD, iD, rD, r2D, pD, nD, xD, yD = reg_stats(df['total_tracks'], df['phq8_change_T2'])
x_lineD = np.linspace(xD.min(), xD.max(), 200)
y_lineD = sD * x_lineD + iD

# ── Panel E: mean_completion vs PHQ-8 change
sE, iE, rE, r2E, pE, nE, xE, yE = reg_stats(df['mean_completion'], df['phq8_change_T2'])
x_lineE = np.linspace(xE.min(), xE.max(), 200)
y_lineE = sE * x_lineE + iE

# ── Panel F: Standardised multiple regression
predictors = ['total_tracks', 'mean_completion', 'days_active',
              'listening_span_days', 'full_completion_rate']
predictors = [p for p in predictors if p in df.columns]

reg_df = df[predictors + ['gad7_change_T2']].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(reg_df[predictors])
y_reg = reg_df['gad7_change_T2'].values


model = LinearRegression().fit(X_scaled, y_reg)
r2_multi = r2_score(y_reg, model.predict(X_scaled))
coefficients = pd.Series(model.coef_, index=predictors).sort_values()

# Significance via scipy for each predictor (simple)
coef_colors = [TEAL if v < 0 else RED for v in coefficients.values]

print(f"\nMultiple regression R² = {r2_multi:.3f}")
print(f"GAD-7 ~ total_tracks:     r={rA:.3f}, R²={r2A:.3f}, p={pA:.4f}")
print(f"GAD-7 ~ mean_completion:  r={rB:.3f}, R²={r2B:.3f}, p={pB:.4f}")
print(f"PHQ-8 ~ total_tracks:     r={rD:.3f}, R²={r2D:.3f}, p={pD:.4f}")
print(f"PHQ-8 ~ mean_completion:  r={rE:.3f}, R²={r2E:.3f}, p={pE:.4f}")

# ── Build figure
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Tracks Listened → GAD-7 Reduction',
        'Completion Rate → GAD-7 Reduction',
        'GAD-7 Change by Dose Group',
        'Tracks Listened → PHQ-8 Reduction',
        'Completion Rate → PHQ-8 Reduction',
        'Standardised Regression Coefficients<br>(GAD-7 outcome)',
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.1,
)

# ── A: tracks vs GAD-7
fig.add_trace(go.Scatter(
    x=xA, y=yA, mode='markers',
    marker=dict(color=BLUE, size=5, opacity=0.25),
    name='Listeners', showlegend=False,
    hovertemplate='Tracks: %{x:.0f}<br>GAD-7 change: %{y:.1f}<extra></extra>',
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=x_lineA, y=y_lineA, mode='lines',
    line=dict(color=TEAL, width=2.5),
    name=f'r={rA:.2f}, R²={r2A:.2f}, p={pA:.3f}',
    showlegend=False, hoverinfo='skip',
), row=1, col=1)
fig.add_annotation(
    text=f'r = {rA:.2f} | R² = {r2A:.2f} | p = {pA:.3f}<br>n = {nA}',
    xref='x1', yref='paper', x=xA.max() * 0.6, y=0.97,
    showarrow=False, font=dict(size=9, color=TEAL),
    bgcolor='rgba(255,255,255,0.8)', bordercolor=TEAL, borderwidth=1,
)

# ── B: completion vs GAD-7
fig.add_trace(go.Scatter(
    x=xB, y=yB, mode='markers',
    marker=dict(color=BLUE, size=5, opacity=0.25),
    showlegend=False,
    hovertemplate='Completion: %{x:.0%}<br>GAD-7 change: %{y:.1f}<extra></extra>',
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=x_lineB, y=y_lineB, mode='lines',
    line=dict(color=TEAL, width=2.5),
    showlegend=False, hoverinfo='skip',
), row=1, col=2)
fig.add_annotation(
    text=f'r = {rB:.2f} | R² = {r2B:.2f} | p = {pB:.3f}<br>n = {nB}',
    xref='x2', yref='paper', x=xB.max() * 0.55, y=0.97,
    showarrow=False, font=dict(size=9, color=TEAL),
    bgcolor='rgba(255,255,255,0.8)', bordercolor=TEAL, borderwidth=1,
)

# ── C: dose group bars
fig.add_trace(go.Bar(
    x=dose_means['dose_group'].astype(str),
    y=dose_means['gad7_change'],
    marker_color=[TEAL if v < 0 else RED for v in dose_means['gad7_change']],
    text=dose_means['gad7_change'].round(1),
    textposition='outside',
    customdata=dose_means['n'],
    hovertemplate='Dose: %{x} tracks<br>GAD-7 change: %{y:.1f}<br>n = %{customdata}<extra></extra>',
    showlegend=False,
), row=1, col=3)
fig.add_hline(y=-3, line_dash='dash', line_color=RED,
              annotation_text='MID (−3)', annotation_font_size=8, row=1, col=3)
fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=3)

# ── D: tracks vs PHQ-8
fig.add_trace(go.Scatter(
    x=xD, y=yD, mode='markers',
    marker=dict(color=AMBER, size=5, opacity=0.25),
    showlegend=False,
    hovertemplate='Tracks: %{x:.0f}<br>PHQ-8 change: %{y:.1f}<extra></extra>',
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=x_lineD, y=y_lineD, mode='lines',
    line=dict(color=AMBER, width=2.5),
    showlegend=False, hoverinfo='skip',
), row=2, col=1)
fig.add_annotation(
    text=f'r = {rD:.2f} | R² = {r2D:.2f} | p = {pD:.3f}<br>n = {nD}',
    xref='x4', yref='paper', x=xD.max() * 0.6, y=0.47,
    showarrow=False, font=dict(size=9, color=AMBER),
    bgcolor='rgba(255,255,255,0.8)', bordercolor=AMBER, borderwidth=1,
)

# ── E: completion vs PHQ-8
fig.add_trace(go.Scatter(
    x=xE, y=yE, mode='markers',
    marker=dict(color=AMBER, size=5, opacity=0.25),
    showlegend=False,
    hovertemplate='Completion: %{x:.0%}<br>PHQ-8 change: %{y:.1f}<extra></extra>',
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=x_lineE, y=y_lineE, mode='lines',
    line=dict(color=AMBER, width=2.5),
    showlegend=False, hoverinfo='skip',
), row=2, col=2)
fig.add_annotation(
    text=f'r = {rE:.2f} | R² = {r2E:.2f} | p = {pE:.3f}<br>n = {nE}',
    xref='x5', yref='paper', x=xE.max() * 0.55, y=0.47,
    showarrow=False, font=dict(size=9, color=AMBER),
    bgcolor='rgba(255,255,255,0.8)', bordercolor=AMBER, borderwidth=1,
)

# ── F: standardised coefficients
fig.add_trace(go.Bar(
    x=coefficients.values,
    y=coefficients.index,
    orientation='h',
    marker_color=coef_colors,
    text=coefficients.round(3).values,
    textposition='outside',
    hovertemplate='%{y}<br>β = %{x:.3f}<extra></extra>',
    showlegend=False,
), row=2, col=3)
fig.add_vline(x=0, line_color='black', line_width=0.8, row=2, col=3)
fig.add_annotation(
    text=f'Multiple R² = {r2_multi:.3f}',
    xref='paper', yref='paper', x=0.99, y=0.03,
    showarrow=False, font=dict(size=10, color='black', family='Arial Bold'),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=GREY, borderwidth=1,
    xanchor='right',
)

# update layout
fig.update_layout(
    title=dict(
        text='<b>Dose-Response Relationship: Engagement as a Predictor of Clinical Change</b>',
        font=dict(size=13), x=0.5,
    ),
    height=750,
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial', size=10),
    margin=dict(t=100, b=60, l=80, r=40),
)

fig.update_xaxes(showgrid=False, linecolor='lightgrey')
fig.update_yaxes(showgrid=True, gridcolor='#f0f0f0', linecolor='lightgrey')

# Axis labels
fig.update_xaxes(title_text='Total tracks listened',  row=1, col=1)
fig.update_xaxes(title_text='Mean completion rate',   row=1, col=2)
fig.update_xaxes(title_text='Dose group (tracks)',    row=1, col=3)
fig.update_xaxes(title_text='Total tracks listened',  row=2, col=1)
fig.update_xaxes(title_text='Mean completion rate',   row=2, col=2)
fig.update_xaxes(title_text='Standardised β',         row=2, col=3)

fig.update_yaxes(title_text='GAD-7 change at T2',  row=1, col=1)
fig.update_yaxes(title_text='GAD-7 change at T2',  row=1, col=2)
fig.update_yaxes(title_text='Mean GAD-7 change',   row=1, col=3)
fig.update_yaxes(title_text='PHQ-8 change at T2',  row=2, col=1)
fig.update_yaxes(title_text='PHQ-8 change at T2',  row=2, col=2)

fig.show()


Multiple regression R² = 0.007
GAD-7 ~ total_tracks:     r=-0.055, R²=0.003, p=0.1007
GAD-7 ~ mean_completion:  r=-0.071, R²=0.005, p=0.0356
PHQ-8 ~ total_tracks:     r=-0.010, R²=0.000, p=0.7605
PHQ-8 ~ mean_completion:  r=-0.060, R²=0.004, p=0.0788


---
# 11. The Free route




In [45]:
# Exclude internal routes
EXCLUDE = ["INTERNAL", "RESEARCH", "MARKETTING"]

analysis_df = listeners_enriched[
    ~listeners_enriched["code_category"].isin(EXCLUDE)
].copy()

all_resp_filtered = response[
    ~response["code_category"].isin(EXCLUDE)
].copy()

# Users who engaged at least once
engaged_ids = set(tracks["id"].unique())

# Ghost users = responded but never engaged
all_resp_filtered["is_ghost"] = ~all_resp_filtered["id"].isin(engaged_ids)

ghost_by_route = (
    all_resp_filtered
    .groupby("code_category")["is_ghost"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

ghost_by_route.columns = ["code_category", "ghost_rate"]


# Calculate change scores for GAD-7 and PHQ-8 at T2
for measure, col0, col2 in [
    ("gad7", "q_gad7_0", "q_gad7_2"),
    ("phq8", "q_phq8_0", "q_phq8_2"),
]:
    if f"{measure}_change_T2" not in analysis_df.columns:
        if col0 in analysis_df.columns and col2 in analysis_df.columns:
            analysis_df[f"{measure}_change_T2"] = (
                pd.to_numeric(analysis_df[col2], errors="coerce") -
                pd.to_numeric(analysis_df[col0], errors="coerce")
            )
            print(f"Created {measure}_change_T2")
        else:
            print(f"Missing columns for {measure}: looked for {col0} and {col2}")
            print(f"Available: {[c for c in analysis_df.columns if measure in c]}")


# Route summary
route_summary = (
    analysis_df
    .groupby("code_category")
    .agg(
        n=("id", "count"),
        baseline_gad7=("q_gad7_0", "mean"),
        baseline_phq8=("q_phq8_0", "mean"),
        gad7_change=("gad7_change_T2", "mean"),
        phq8_change=("phq8_change_T2", "mean"),
        mean_tracks=("total_tracks", "mean"),
        mean_completion=("mean_completion", "mean"),
    )
    .round(2)
    .reset_index()
)

route_summary = route_summary.merge(
    ghost_by_route,
    on="code_category",
    how="left"
)

routes = route_summary["code_category"].tolist()
bar_colors = [AMBER if r == "FREE" else BLUE for r in routes]

coloured_ticks = dict(
    tickmode="array",
    tickvals=routes,
    ticktext=[
        f'<span style="color:{"#B8960A" if r == "PROMOTION" else color_map_referral_codes.get(r, "#333333")}"><b>{r}</b></span>'
        for r in routes
    ],
    tickangle=-45,
    tickfont=dict(size=10, family="Arial"),
)

In [46]:
# Baseline GAD-7 by route of access
fig = go.Figure()

fig.add_trace(go.Bar(
    x=routes,
    y=route_summary["baseline_gad7"],
    marker_color=bar_colors,
    text=route_summary["baseline_gad7"].round(1),
    textposition="outside",
    name="Baseline GAD-7",
    showlegend=False,
    hovertemplate="%{x}<br>Baseline GAD-7: %{y:.1f}<extra></extra>",
))

fig.add_hline(
    y=10,
    line_dash="dash",
    line_color=RED,
    annotation_text="Clinical threshold (10)",
    annotation_font_size=9,
)

fig.update_layout(
    title=dict(
        text="<b>Baseline GAD-7 Severity by Route</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=80, l=60, r=40),
)

fig.update_xaxes(**coloured_ticks, showgrid=False, linecolor="lightgrey")
fig.update_yaxes(
    title_text="Mean GAD-7 score",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()

In [47]:
# Ghost rate by route
fig = go.Figure()

fig.add_trace(go.Bar(
    x=routes,
    y=route_summary["ghost_rate"],
    marker_color=bar_colors,
    text=route_summary["ghost_rate"].round(1).astype(str) + "%",
    textposition="outside",
    name="Ghost Rate",
    showlegend=False,
    hovertemplate="%{x}<br>Ghost rate: %{y:.1f}%<extra></extra>",
))

fig.update_layout(
    title=dict(
        text="<b>Ghost Rate: Users Who Never Engaged</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=80, l=60, r=40),
)

fig.update_xaxes(**coloured_ticks, showgrid=False, linecolor="lightgrey")
fig.update_yaxes(
    title_text="% users never engaging",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()

In [48]:
# Mean tracks listened by route
fig = go.Figure()

fig.add_trace(go.Bar(
    x=routes,
    y=route_summary["mean_tracks"],
    marker_color=bar_colors,
    text=route_summary["mean_tracks"].round(1),
    textposition="outside",
    name="Mean Tracks",
    showlegend=False,
    hovertemplate="%{x}<br>Mean tracks: %{y:.1f}<extra></extra>",
))

fig.update_layout(
    title=dict(
        text="<b>Mean Tracks Listened by Route</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=80, l=60, r=40),
)

fig.update_xaxes(**coloured_ticks, showgrid=False, linecolor="lightgrey")
fig.update_yaxes(
    title_text="Mean tracks",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()

In [49]:
# Mean GAD-7 change by route of access among sustained engagers
change_colors = [
    TEAL if v < 0 else RED
    for v in route_summary["gad7_change"]
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=routes,
    y=route_summary["gad7_change"],
    marker_color=change_colors,
    text=route_summary["gad7_change"].round(1),
    textposition="outside",
    name="GAD-7 Change",
    showlegend=False,
    hovertemplate="%{x}<br>GAD-7 change: %{y:.1f}<extra></extra>",
))

fig.add_hline(
    y=-3,
    line_dash="dash",
    line_color=RED,
    annotation_text="MID threshold (−3)",
    annotation_font_size=9,
)

fig.add_hline(
    y=0,
    line_color="black",
    line_width=0.8,
)

fig.update_layout(
    title=dict(
        text="<b>Mean GAD-7 Change at T2</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=80, l=60, r=40),
)

fig.update_xaxes(**coloured_ticks, showgrid=False, linecolor="lightgrey")
fig.update_yaxes(
    title_text="Mean score change",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()

In [50]:
# Regression-to-mean analysis for GAD-7
rtm_df = analysis_df[["q_gad7_0", "gad7_change_T2"]].dropna()

slope, intercept, r_val, p_val, _ = stats.linregress(
    rtm_df["q_gad7_0"],
    rtm_df["gad7_change_T2"]
)

x_line = np.linspace(
    rtm_df["q_gad7_0"].min(),
    rtm_df["q_gad7_0"].max(),
    100
)

y_line = slope * x_line + intercept

print(
    f"Regression-to-mean (GAD-7): "
    f"r = {r_val:.3f}, p = {p_val:.4f}, slope = {slope:.3f}"
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rtm_df["q_gad7_0"],
    y=rtm_df["gad7_change_T2"],
    mode="markers",
    marker=dict(color=GREY, size=5, opacity=0.3),
    name="Individual listeners",
    showlegend=False,
    hovertemplate="Baseline: %{x:.0f}<br>Change: %{y:.1f}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    line=dict(color=RED, width=2),
    name=f"Trend (r={r_val:.2f}, p={p_val:.3f})",
    showlegend=False,
    hoverinfo="skip",
))

# FREE route mean overlay
if "FREE" in route_summary["code_category"].values:
    free_row = route_summary[
        route_summary["code_category"] == "FREE"
    ].iloc[0]

    fig.add_trace(go.Scatter(
        x=[free_row["baseline_gad7"]],
        y=[free_row["gad7_change"]],
        mode="markers+text",
        marker=dict(
            color=AMBER,
            size=14,
            symbol="diamond",
            line=dict(color="white", width=1.5),
        ),
        text=["FREE mean"],
        textposition="top right",
        textfont=dict(size=9),
        name="FREE route mean",
        showlegend=False,
        hovertemplate=(
            "FREE mean<br>"
            "Baseline: %{x:.1f}<br>"
            "Change: %{y:.1f}<extra></extra>"
        ),
    ))

fig.add_hline(
    y=0,
    line_color="black",
    line_width=0.6,
)

fig.update_layout(
    title=dict(
        text="<b>Baseline Severity vs GAD-7 Change</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=70, l=70, r=40),
)

fig.update_xaxes(
    title_text="Baseline GAD-7 severity",
    showgrid=False,
    linecolor="lightgrey",
)

fig.update_yaxes(
    title_text="GAD-7 change at T2",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()

Regression-to-mean (GAD-7): r = -0.488, p = 0.0000, slope = -0.439


In [51]:
# Bubble plot: baseline severity vs ghost rate
bubble_sizes = (
    route_summary["n"] / route_summary["n"].max() * 60
).tolist()

bubble_colors = [
    AMBER if r == "FREE" else BLUE
    for r in routes
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=route_summary["baseline_gad7"],
    y=route_summary["ghost_rate"],
    mode="markers+text",
    marker=dict(
        size=bubble_sizes,
        color=bubble_colors,
        opacity=0.75,
        line=dict(color="white", width=1.5),
        sizemode="diameter",
    ),
    text=routes,
    textposition="top right",
    textfont=dict(size=9),
    customdata=np.stack(
        [
            route_summary["n"],
            route_summary["baseline_gad7"],
        ],
        axis=-1,
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Baseline GAD-7: %{x:.1f}<br>"
        "Ghost rate: %{y:.1f}%<br>"
        "n users: %{customdata[0]:.0f}<extra></extra>"
    ),
    showlegend=False,
))

fig.add_annotation(
    text="⚠ Top-right = highest risk:<br>high severity, low activation",
    xref="paper",
    yref="paper",
    x=0.99,
    y=0.02,
    xanchor="right",
    yanchor="bottom",
    showarrow=False,
    bgcolor="#fff3cd",
    bordercolor=AMBER,
    borderwidth=1,
    font=dict(size=10, color=RED),
)

fig.update_layout(
    title=dict(
        text="<b>Severity vs Non-Activation Rate</b>",
        font=dict(size=16),
        x=0.5,
    ),
    height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=11),
    margin=dict(t=80, b=70, l=70, r=40),
)

fig.update_xaxes(
    title_text="Mean baseline GAD-7",
    showgrid=False,
    linecolor="lightgrey",
)

fig.update_yaxes(
    title_text="Ghost rate (%)",
    showgrid=True,
    gridcolor="#f0f0f0",
    linecolor="lightgrey",
)

fig.show()